# 🤖 Transformer & Multi-Head Attention — từ trực giác, công thức, code đến LLM hiện đại và tiếng Việt

**Biên soạn và giảng dạy:** TS. Hoàng Văn Quý
**Học phần:** Deep Learning trong NLP · **Phiên bản:** 2.0 (cập nhật 09/2026, chạy trên Google Colab)

> Phần cài đặt kế thừa **The Annotated Transformer** (Rush, 2018; bản cập nhật 2022 của Huang, Subramanian, Sum, Almubarak & Biderman) và phần trực giác kế thừa bài giảng **"Attention in transformers, step-by-step"** của 3Blue1Brown (Sanderson, 2024).

Notebook dẫn dắt **từ trực giác → công thức → ví dụ số → code chạy được**, cài đặt toàn bộ kiến trúc Transformer gốc (Vaswani et al., 2017), sau đó trả lời câu hỏi mà các LLM 2023–2026 phải đối mặt: *attention tốn bao nhiêu, và mỗi thế hệ cắt chi phí ở đâu?* Cuối cùng, áp dụng cho **tiếng Việt**: âm tiết và từ, chuẩn hoá dấu, "thuế token", và huấn luyện một Transformer tự cài để **khôi phục dấu tiếng Việt**.

---

## 🎯 Mục tiêu học tập

Sau notebook này, sinh viên có thể:
1. **Giải thích bằng trực giác và bằng số** cơ chế Attention: Q, K, V là gì, vì sao chia $\sqrt{d_k}$, vì sao cần mask.
2. **Cài đặt từng thành phần** của Transformer bằng PyTorch và lắp ráp một mô hình Encoder–Decoder huấn luyện được.
3. **Đếm tham số và bộ nhớ**: tham số GPT-3, KV-cache của Llama-3-8B/DeepSeek-V3, chi phí $O(T^2)$.
4. **Nắm các cải tiến 2019–2026**: RoPE, ALiBi, GQA/MLA, FlashAttention, sliding-window, linear attention, RMSNorm, SwiGLU, QK-Norm, gated attention.
5. **Áp dụng cho tiếng Việt**: hiểu ảnh hưởng của tách từ, chuẩn hoá Unicode, tokenizer lên chi phí attention; tự huấn luyện mô hình khôi phục dấu.

---

## 📚 Mục lục

| Phần | Nội dung | Trọng tâm |
|------|----------|-----------|
| **A** | Cài đặt môi trường | Setup |
| **B** | Tổng quan kiến trúc Transformer | Bức tranh lớn, bối cảnh lịch sử |
| **C** | Token Embedding & Positional Encoding | Biểu diễn đầu vào |
| **D** | Trực giác Attention (theo 3B1B) + **ví dụ tính tay** | Q / K / V là gì |
| **E** | Scaled Dot-Product Attention | Công thức cốt lõi, vì sao $\sqrt{d_k}$ |
| **F** | Multi-Head Attention | $h$ "đầu" song song |
| **G** | Position-wise Feed-Forward Network | Biến đổi phi tuyến |
| **H** | LayerNorm, RMSNorm & Residual | Ổn định huấn luyện, pre-LN vs post-LN |
| **I** | Encoder | Khối mã hoá |
| **J** | Decoder & Cross-Attention | Khối giải mã |
| **K** | Lắp ráp Full Transformer | Mô hình hoàn chỉnh |
| **L** | Đếm tham số (Transformer-base, GPT-3) | Hiểu quy mô |
| **M** | Huấn luyện: Batch, Mask, Optimizer, Label Smoothing | Thực hành |
| **N** | Demo: Synthetic Copy Task | Kiểm chứng end-to-end |
| **P** | 🆕 **Attention hiện đại 2019–2026** | KV-cache, GQA/MLA, RoPE/ALiBi, FlashAttention, sparse/linear, khối LLM 2025 |
| **Q** | 🆕 **Ứng dụng cho tiếng Việt** | Tách từ, chuẩn hoá, thuế token, **khôi phục dấu bằng Transformer tự cài** |
| **R** | 🆕 Bài tập & câu hỏi thảo luận | Củng cố, đề tài mở |
| **O** | Tổng kết & hướng đi tiếp theo | Bước kế tiếp |
| **—** | 🆕 Tài liệu tham khảo | Trích dẫn đầy đủ |

**Quy ước:** mỗi phần mở đầu bằng khung **⏱ Ý chính** (đọc trong 30 giây). Ký hiệu *[preprint]* = tài liệu chưa qua bình duyệt (arXiv, báo cáo kỹ thuật, model card). Các con số ghi *(tính toán)* là phép tính từ cấu hình đã dẫn, không phải số đo.

---

## ⚙️ Lưu ý khi chạy trên Google Colab

- **Runtime → Change runtime type → GPU** (T4) giúp nhanh hơn nhưng **không bắt buộc**: mọi ví dụ được thiết kế chạy trên CPU.
- Chạy lần lượt **từ trên xuống** (cell sau phụ thuộc cell trước).
- Các cell có nhãn `# [cần Internet]` tải mô hình/dữ liệu từ Hugging Face; nếu không có mạng, notebook bỏ qua và chạy tiếp.
- Tổng thời gian chạy trọn vẹn: ~6–10 phút trên CPU (phần lớn cho hai lần huấn luyện ở N và Q.5).

---

# 🔧 A. Cài đặt môi trường

Trên Colab, các thư viện cần thiết đã có sẵn. Nếu chạy local, bỏ comment dòng `pip install`.

In [ ]:
# Nếu chạy local và chưa cài: bỏ comment dòng dưới
# !pip install torch numpy matplotlib

import math
import copy
import time
import random
import re
import unicodedata
from collections import Counter, defaultdict
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle, FancyBboxPatch

# Cố định seed để kết quả tái lập được
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)

# Thiết bị (các ví dụ nhỏ trong notebook chạy trên CPU để kết quả ổn định)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Thiết lập matplotlib
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.unicode_minus'] = False

print(f"PyTorch: {torch.__version__}")
print(f"NumPy  : {np.__version__}")
print(f"Device : {device}")
# Ghi chú: bản gốc import torch.autograd.Variable — API này đã lỗi thời từ PyTorch 0.4, nay không cần.

**Các thư viện được sử dụng:**

| Thư viện | Mục đích |
|----------|---------|
| `math`, `copy`, `time`, `random` | Toán cơ bản, sao chép sâu các module, đo thời gian, lấy mẫu |
| `re`, `unicodedata` | Xử lý chuỗi và chuẩn hoá Unicode tiếng Việt (Phần Q) |
| `numpy` | Mảng số học (tạo mask, tính toán đơn giản) |
| `torch`, `torch.nn`, `torch.nn.functional` | Cốt lõi PyTorch để xây mạng nơ-ron |
| `matplotlib` | Vẽ tất cả các hình minh hoạ trong notebook |

---

# 🏛️ B. Tổng quan kiến trúc Transformer

> **⏱ Ý chính.** RNN đọc từng token một nên chậm và "quên" thông tin xa. Attention cho mỗi token **nhìn trực tiếp** mọi token khác trong một bước. Transformer bỏ hẳn RNN, chỉ giữ attention + mạng feed-forward, nên song song hoá được hoàn toàn.

## B.1. Vì sao lại có Transformer?

Trước 2017, dịch máy nơ-ron và hầu hết bài toán NLP dùng **RNN/LSTM** — xử lý chuỗi *tuần tự* từng token. Hạn chế:

1. **Khó song song hoá** — bước $t$ phải chờ bước $t-1$.
2. **Gradient vanishing** với chuỗi dài — thông tin từ token đầu khó "đi xa" tới token cuối.
3. **Nút cổ chai vector cố định** — toàn bộ câu nguồn bị nén vào một vector. Bahdanau, Cho & Bengio (2015) viết rằng họ *phỏng đoán* vector độ dài cố định là nút cổ chai, và đề xuất để decoder tự "(soft-)search" các vị trí liên quan ở câu nguồn — đó là **attention** đầu tiên trong dịch máy (additive attention). Luong, Pham & Manning (2015) đề xuất dạng **nhân (multiplicative/dot-product)** đơn giản hơn.

Vaswani et al. (2017), *"Attention Is All You Need"*, loại bỏ hoàn toàn RNN/CNN. Kết quả trên WMT 2014:

| Mô hình | $N$ | $d_{\text{model}}$ | $d_{ff}$ | $h$ | Tham số | BLEU En→De | BLEU En→Fr | Huấn luyện (8×P100) |
|---|---|---|---|---|---|---|---|---|
| Transformer-base | 6 | 512 | 2048 | 8 | 65M | 27.3 | 38.1 | 12 giờ (100K bước) |
| Transformer-big | 6 | 1024 | 4096 | 16 | 213M | 28.4 | 41.8 | 3,5 ngày (300K bước) |

*Nguồn: Vaswani et al. (2017), Table 2–3.* Điểm then chốt không chỉ là BLEU cao hơn mà là **chi phí huấn luyện thấp hơn nhiều** so với các mô hình RNN/CNN tốt nhất thời đó — mở đường cho việc scale lên hàng trăm tỉ tham số.

## B.2. Bức tranh lớn

Transformer có hai khối chính:

- **Encoder** — đọc và mã hoá chuỗi nguồn $(x_1, \dots, x_n)$ thành biểu diễn liên tục $\mathbf{z} = (z_1, \dots, z_n)$.
- **Decoder** — dựa vào $\mathbf{z}$ và các token đã sinh trước đó, sinh từng token đầu ra $(y_1, \dots, y_m)$ theo kiểu **auto-regressive**.

Mỗi khối xếp chồng $N$ tầng giống hệt nhau (paper gốc: $N=6$). Sơ đồ:

In [ ]:
# Sơ đồ tổng quan kiến trúc Transformer
fig, ax = plt.subplots(figsize=(12, 9))
ax.set_xlim(0, 10); ax.set_ylim(0, 12)
ax.axis('off')

def block(x, y, w, h, label, color, fontsize=10):
    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                          linewidth=1.5, edgecolor='black', facecolor=color)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center', fontsize=fontsize, weight='bold')

def arrow(x1, y1, x2, y2, color='black'):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='->',
                                  mutation_scale=15, color=color, linewidth=1.5))

# ----- Bên trái: ENCODER -----
block(0.5, 0.5, 3, 0.7, 'Input Embedding + PE', '#FFE4B5')
arrow(2, 1.2, 2, 1.8)
# Encoder Layer
block(0.3, 1.8, 3.4, 4.5, '', '#E8F4F8')
ax.text(2, 6.0, 'Encoder × N', fontsize=11, weight='bold', ha='center', style='italic')
block(0.6, 4.5, 2.8, 0.8, 'Multi-Head Self-Attention', '#B0E0E6')
block(0.6, 3.3, 2.8, 0.8, 'Add & Norm', '#D3D3D3')
block(0.6, 2.1, 2.8, 0.8, 'Feed-Forward', '#B0E0E6')
arrow(2, 5.3, 2, 5.5)  # MHA -> norm above
block(0.6, 5.5, 2.8, 0.5, 'Add & Norm', '#D3D3D3')
# arrows inside encoder
arrow(2, 1.8, 2, 2.1)
arrow(2, 2.9, 2, 3.3)
arrow(2, 4.1, 2, 4.5)
arrow(2, 6.0, 2, 6.3)

# Memory đi sang decoder
ax.annotate('', xy=(5.5, 6.0), xytext=(3.7, 6.0),
            arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.text(4.6, 6.2, 'memory', fontsize=9, color='red', ha='center', style='italic')

# ----- Bên phải: DECODER -----
block(6.5, 0.5, 3, 0.7, 'Output Embedding + PE\n(shifted right)', '#FFE4B5')
arrow(8, 1.2, 8, 1.8)
block(6.3, 1.8, 3.4, 6.5, '', '#FFF0F5')
ax.text(8, 8.0, 'Decoder × N', fontsize=11, weight='bold', ha='center', style='italic')
block(6.6, 6.7, 2.8, 0.8, 'Masked Multi-Head\nSelf-Attention', '#FFC0CB')
block(6.6, 5.5, 2.8, 0.5, 'Add & Norm', '#D3D3D3')
block(6.6, 4.3, 2.8, 0.8, 'Cross-Attention\n(với memory)', '#FFB6C1')
block(6.6, 3.1, 2.8, 0.5, 'Add & Norm', '#D3D3D3')
block(6.6, 1.9, 2.8, 0.8, 'Feed-Forward', '#FFC0CB')
arrow(8, 1.8, 8, 1.9)
arrow(8, 2.7, 8, 3.1)
arrow(8, 3.6, 8, 4.3)
arrow(8, 5.1, 8, 5.5)
arrow(8, 6.0, 8, 6.7)
arrow(8, 7.5, 8, 8.0)

# Linear + Softmax
block(6.5, 9.0, 3, 0.7, 'Linear + Softmax', '#FFE4B5')
arrow(8, 8.3, 8, 9.0)
arrow(8, 9.7, 8, 10.4)
ax.text(8, 10.7, 'Output Probabilities', fontsize=10, weight='bold', ha='center')

ax.set_title('Kiến trúc Transformer (Vaswani et al., 2017)', fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

**Đọc sơ đồ trên:**

- **Bên trái (Encoder):** đọc chuỗi nguồn. Mỗi tầng gồm 2 sub-layer: **Self-Attention** (mỗi token nhìn vào các token khác trong nguồn) + **Feed-Forward**.
- **Bên phải (Decoder):** sinh chuỗi đích. Mỗi tầng gồm 3 sub-layer: **Masked Self-Attention** (chỉ nhìn token trước trong đích) + **Cross-Attention** (nhìn vào output của encoder) + **Feed-Forward**.
- **Mũi tên đỏ** (`memory`): biểu diễn ẩn của encoder được "tiêm" vào mọi tầng decoder thông qua cross-attention.
- Mỗi sub-layer có **Add & Norm** (residual + LayerNorm) để ổn định gradient khi mạng sâu. Lưu ý: *bài báo gốc* đặt LayerNorm **sau** phép cộng (post-LN); *code* ở phần H đặt **trước** (pre-LN) — xem giải thích ở H.2.

Notebook này sẽ cài đặt **từng khối nhỏ một**, theo thứ tự "từ trong ra ngoài" rồi gộp lại.

---

# 🔤 C. Token Embedding & Positional Encoding

> **⏱ Ý chính.** Token → số nguyên → vector (embedding). Vì attention không biết thứ tự, ta cộng thêm một vector "vị trí". Bản gốc dùng hàm sin/cos; LLM hiện đại dùng RoPE hoặc ALiBi (Phần P).

## C.1. Token Embedding — "biến chữ thành số"

Mô hình không hiểu chữ; nó chỉ hiểu **số**. Mỗi token (có chỉ số $0, 1, \dots, V-1$ trong từ điển) được ánh xạ thành một **vector** $d_{\text{model}}$ chiều — gọi là **embedding**.

`nn.Embedding(vocab_size, d_model)` bản chất là một **bảng tra cứu**: chỉ số $i$ → hàng thứ $i$ của ma trận trọng số.

**Ví dụ số:** vocab $V = 37\,000$ (BPE dùng chung cho Anh–Đức trong Vaswani et al., 2017), $d_{\text{model}} = 512$ → bảng embedding có $37\,000 \times 512 \approx 18{,}9$ triệu tham số — gần 30% của Transformer-base. Vì thế paper gốc **dùng chung (weight tying)** một ma trận cho embedding nguồn, embedding đích và lớp Linear trước softmax (Vaswani et al., 2017, §3.4; Press & Wolf, 2017).

Paper gốc còn nhân embedding với $\sqrt{d_{\text{model}}}$ để độ lớn của embedding tương xứng với Positional Encoding (có giá trị trong $[-1, 1]$):

In [ ]:
class Embeddings(nn.Module):
    """Lớp embedding token, nhân với sqrt(d_model) theo paper gốc."""

    def __init__(self, d_model: int, vocab: int):
        super().__init__()
        self.lut = nn.Embedding(vocab, d_model)  # 'lut' = look-up table
        self.d_model = d_model

    def forward(self, x):
        # x: tensor int64 shape (batch, seq_len)
        # output: (batch, seq_len, d_model)
        return self.lut(x) * math.sqrt(self.d_model)


# Demo
emb = Embeddings(d_model=64, vocab=100)
tokens = torch.tensor([[1, 5, 23, 7]])  # 1 câu, 4 token
print(f"Input token ids: {tokens.tolist()}")
print(f"Embedding shape: {tuple(emb(tokens).shape)}  # (batch, seq, d_model)")

## C.2. Positional Encoding — "vị trí cũng quan trọng"

Self-Attention thuần tuý **không phân biệt thứ tự** — với nó, *"Tôi yêu em"* và *"Em yêu tôi"* là cùng một *tập* token, nên cho cùng tập biểu diễn (chỉ bị hoán vị).

Paper gốc **cộng thêm** một vector vị trí vào embedding, dùng hàm sin/cos với các tần số khác nhau:

$$
\begin{aligned}
PE_{(pos,\; 2i)} &= \sin\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right) \\
PE_{(pos,\; 2i+1)} &= \cos\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
\end{aligned}
$$

**Ví dụ số** ($d_{\text{model}} = 4$, vị trí $pos = 1$): tần số của cặp chiều 0–1 là $1/10000^{0} = 1$, của cặp 2–3 là $1/10000^{2/4} = 0{,}01$. Vậy
$PE_1 = [\sin 1,\ \cos 1,\ \sin 0{,}01,\ \cos 0{,}01] \approx [0{,}841,\ 0{,}540,\ 0{,}010,\ 1{,}000]$.
Cặp đầu "quay" nhanh (1 rad/vị trí), cặp sau quay chậm (0,01 rad/vị trí) — giống kim giây và kim giờ của đồng hồ.

**Tại sao chọn sin/cos?**

- Mỗi cặp chiều có **tần số khác nhau** → vị trí được biểu diễn ở nhiều "tỉ lệ" (thô đến mịn).
- Với mọi khoảng cách $k$, $PE_{pos+k}$ là một **phép quay** (biến đổi tuyến tính) của $PE_{pos}$ → tích vô hướng $PE_{pos}\cdot PE_{pos+k}$ chỉ phụ thuộc $k$ (kiểm chứng bằng code bên dưới). Đây chính là mầm mống của **RoPE** (Phần P.5).
- ⚠️ **Đính chính một hiểu lầm phổ biến:** Vaswani et al. (2017) chỉ *hy vọng* sin/cos "có thể" giúp ngoại suy ra chuỗi dài hơn lúc huấn luyện, và báo cáo rằng PE *học được* cho kết quả "gần như giống hệt" (Table 3, dòng E). Thực nghiệm sau này cho thấy sinusoidal PE **ngoại suy rất kém**: chất lượng chỉ giữ được thêm khoảng 20–50 token sau độ dài huấn luyện (Press, Smith & Lewis, 2022). Đây là động lực cho ALiBi, RoPE và các kỹ thuật kéo dài ngữ cảnh.

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal Positional Encoding — không có tham số học được."""

    def __init__(self, d_model: int, dropout: float, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Tính trước ma trận PE shape (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()    # (max_len, 1)

        # div_term[i] = 1 / 10000^(2i / d_model), tính qua log để ổn định số
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)   # các chiều CHẴN
        pe[:, 1::2] = torch.cos(position * div_term)   # các chiều LẺ
        pe = pe.unsqueeze(0)                           # (1, max_len, d_model) để broadcast batch

        # register_buffer: lưu trong state_dict nhưng KHÔNG học (không phải Parameter)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model). Cộng PE tương ứng với seq_len.
        x = x + self.pe[:, :x.size(1)].requires_grad_(False)
        return self.dropout(x)


# Trực quan hoá PE
pe_module = PositionalEncoding(d_model=64, dropout=0.0)
zeros = torch.zeros(1, 100, 64)
pe_out = pe_module(zeros).squeeze(0).numpy()    # (100, 64)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# (a) Heatmap toàn bộ PE
im = axes[0].imshow(pe_out, aspect='auto', cmap='RdBu_r')
axes[0].set_xlabel('Chiều embedding (i)')
axes[0].set_ylabel('Vị trí (pos)')
axes[0].set_title('Heatmap Positional Encoding\n(64 chiều × 100 vị trí)')
plt.colorbar(im, ax=axes[0])

# (b) Một vài chiều theo vị trí
for i in [0, 1, 4, 5, 20, 21]:
    axes[1].plot(np.arange(100), pe_out[:, i], label=f'dim {i}')
axes[1].set_xlabel('Vị trí (pos)')
axes[1].set_ylabel('Giá trị PE')
axes[1].set_title('Một số chiều của PE: tần số khác nhau\n(chiều thấp = tần số cao)')
axes[1].legend(ncol=2, fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Nhận xét hai biểu đồ:**

- **Trái (heatmap):** các chiều thấp (bên trái) dao động *nhanh* theo vị trí (sọc sát nhau); chiều cao (bên phải) dao động *chậm* (sọc thưa). Như vậy mỗi vị trí có "vân tay" duy nhất.
- **Phải (curve):** chiều `dim 0`, `dim 1` dao động nhanh (tần số cao); chiều `dim 20`, `dim 21` dao động chậm. Đó là cách PE biểu diễn vị trí ở **nhiều tỉ lệ** cùng lúc, giống cách Fourier phân tích tín hiệu.

Tổng kết khối Embedding:

In [ ]:
# Kiểm chứng: PE_pos · PE_(pos+k) chỉ phụ thuộc khoảng cách k, không phụ thuộc pos
pe_full = PositionalEncoding(d_model=64, dropout=0.0).pe[0]      # (max_len, 64)
for k in [1, 5, 20]:
    dots = [float(pe_full[p] @ pe_full[p + k]) for p in [0, 10, 50, 200]]
    print(f"k = {k:2d}: PE_p·PE_(p+k) tại p = 0, 10, 50, 200 →", [f"{d:.3f}" for d in dots])
print("→ Cùng một k cho cùng giá trị: thông tin 'khoảng cách tương đối' nằm sẵn trong PE.")

In [ ]:
# Kết hợp Embedding + PE thành pipeline đầu vào
d_model, vocab = 64, 100
input_pipeline = nn.Sequential(
    Embeddings(d_model, vocab),
    PositionalEncoding(d_model, dropout=0.1)
)

x = torch.tensor([[5, 12, 7, 3, 99, 0, 0, 0]])    # 1 câu, 8 token (có padding)
out = input_pipeline(x)
print(f"Input  : {tuple(x.shape)}     (batch, seq_len)")
print(f"Output : {tuple(out.shape)}  (batch, seq_len, d_model)")
# Lưu ý: token 0 ở đây là padding; mask ở phần M sẽ đảm bảo attention bỏ qua các vị trí này.

---

# 💡 D. Trực giác về Cơ chế Attention

> **⏱ Ý chính.** Mỗi token hỏi (Query) "ai liên quan đến tôi?", mỗi token khác trả lời bằng nhãn (Key); mức khớp Q·K quyết định lấy bao nhiêu nội dung (Value) của nhau. Kết quả: vector của mỗi từ được "chỉnh" theo ngữ cảnh.

Phần này dựa trên cách giải thích trực quan của 3Blue1Brown (Sanderson, 2024).

## D.1. Tại sao cần Attention?

Embedding ban đầu chỉ phụ thuộc *bản thân từ*, không quan tâm ngữ cảnh. Nhưng nghĩa của từ trong câu phụ thuộc hoàn toàn vào các từ xung quanh:

| Câu tiếng Việt | Nghĩa của *"đường"* |
|----------------|--------------------|
| "Đường này tắc lắm" | con đường (giao thông) |
| "Đường ngọt quá" | gia vị (đồ ngọt) |
| "Đường ống nước bị vỡ" | ống dẫn |

Sau bước embedding, **cả ba** vector của *"đường"* giống hệt nhau. Nhiệm vụ của Attention: cho phép các từ xung quanh *thêm thông tin* để vector của *"đường"* "trượt" sang vùng không gian biểu diễn nghĩa đúng.

> 🇻🇳 Tiếng Việt có rất nhiều hiện tượng như vậy, kể cả ở mức **dấu thanh**: khi mất dấu, âm tiết "duong" có thể là *đường, dương, dường, dưỡng…* — chỉ ngữ cảnh mới quyết định được. Phần Q.5 sẽ dùng chính bài toán này để huấn luyện Transformer.

In [ ]:
# Minh hoạ trực giác: embedding ban đầu vs sau Attention
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# --- Bên trái ---
ax = axes[0]
ax.scatter([0.5], [0.5], s=400, c='steelblue', zorder=3, edgecolors='black')
ax.annotate('"đường"\n(vector chung)', (0.5, 0.5), xytext=(0.55, 0.55),
            fontsize=11, weight='bold')
ax.text(0.5, 0.1, 'Cả 3 câu đều có cùng một vector', ha='center',
        fontsize=10, style='italic', color='gray')
ax.set_title('Trước Attention:\nembedding KHÔNG biết ngữ cảnh', fontsize=12)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel('Trục ẩn 1'); ax.set_ylabel('Trục ẩn 2')
ax.grid(alpha=0.3)

# --- Bên phải ---
ax = axes[1]
ax.scatter([0.5], [0.5], s=200, c='gray', alpha=0.4, zorder=2)
ax.annotate('"đường"\nban đầu', (0.5, 0.5), xytext=(0.42, 0.38),
            fontsize=9, color='gray', style='italic')

# Ba vị trí sau Attention với 3 ngữ cảnh khác nhau
positions = [
    (0.85, 0.75, 'orange', 'gia vị\n(ngữ cảnh: "ngọt")'),
    (0.15, 0.80, 'green', 'giao thông\n(ngữ cảnh: "tắc")'),
    (0.20, 0.20, 'purple', 'ống dẫn\n(ngữ cảnh: "ống nước")'),
]
for x, y, c, label in positions:
    ax.scatter([x], [y], s=300, c=c, zorder=3, edgecolors='black')
    ax.annotate(label, (x, y), xytext=(x + 0.02, y + 0.02),
                fontsize=9, color=c, weight='bold')
    ax.add_patch(FancyArrowPatch((0.5, 0.5), (x, y), arrowstyle='->',
                                  mutation_scale=18, color=c, alpha=0.6, linewidth=2))

ax.set_title('Sau Attention:\nembedding đã TINH CHỈNH theo ngữ cảnh', fontsize=12)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel('Trục ẩn 1'); ax.set_ylabel('Trục ẩn 2')
ax.grid(alpha=0.3)

plt.suptitle('Vai trò của Attention: dịch chuyển embedding theo ngữ cảnh',
             fontsize=13, weight='bold', y=1.02)
plt.tight_layout()
plt.show()

## D.2. Ba vai trò: Query, Key, Value

Hãy hình dung mỗi token đóng **ba vai** cùng lúc:

| Vai | Vector | Câu hỏi/Vai trò |
|-----|--------|----------------|
| **Query** ($\vec{Q}$) | "Câu hỏi" | *"Tôi đang tìm thông tin gì?"* |
| **Key** ($\vec{K}$) | "Thẻ nhận dạng" | *"Tôi có thể trả lời cho câu hỏi như thế nào?"* |
| **Value** ($\vec{V}$) | "Thông tin chia sẻ" | *"Nếu khớp, tôi sẽ đưa thông tin này"* |

**Ví dụ ẩn dụ — thư viện:** Bạn (Query) đến thư viện hỏi: *"Tôi muốn tìm sách về Transformer."* Mỗi cuốn sách trên kệ có một **nhãn dán** (Key) ở gáy. Bạn so sánh câu hỏi với các nhãn — cuốn nào khớp nhất, bạn lấy *nội dung* (Value) của cuốn đó về đọc.

Cách "tạo" Q, K, V từ embedding $\vec{E}$ của một token:

$$
\vec{Q} = W_Q \vec{E}, \quad \vec{K} = W_K \vec{E}, \quad \vec{V} = W_V \vec{E}
$$

Ba ma trận $W_Q, W_K, W_V$ là **tham số học được** — chính là thứ mô hình "học" trong quá trình huấn luyện.

In [ ]:
# Minh hoạ: từ E sinh ra Q, K, V
fig, ax = plt.subplots(figsize=(11, 6))
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')

# Embedding ở giữa-trái
emb_box = FancyBboxPatch((0.5, 2.5), 1.5, 1, boxstyle="round,pad=0.05",
                          facecolor='#FFE4B5', edgecolor='black', linewidth=2)
ax.add_patch(emb_box)
ax.text(1.25, 3.0, r'$\vec{E}$' + '\n(embedding)', ha='center', va='center',
        fontsize=12, weight='bold')

# 3 ma trận chiếu
for i, (y, name, color) in enumerate([(5, '$W_Q$', '#B0E0E6'),
                                        (3, '$W_K$', '#FFB6C1'),
                                        (1, '$W_V$', '#90EE90')]):
    # Ma trận
    mbox = FancyBboxPatch((4, y - 0.4), 1.2, 0.8, boxstyle="round,pad=0.05",
                           facecolor=color, edgecolor='black')
    ax.add_patch(mbox)
    ax.text(4.6, y, name, ha='center', va='center', fontsize=13, weight='bold')
    # Mũi tên từ E
    ax.add_patch(FancyArrowPatch((2.0, 3.0), (4.0, y), arrowstyle='->',
                                  mutation_scale=18, linewidth=1.5))
    # Vector kết quả
    out_label = ['$\\vec{Q}$ (Query)\n"Tôi tìm gì?"',
                 '$\\vec{K}$ (Key)\n"Tôi là gì?"',
                 '$\\vec{V}$ (Value)\n"Tôi mang gì?"'][i]
    out_color = ['#E0F0FF', '#FFE0E8', '#E0FFE0'][i]
    obox = FancyBboxPatch((7, y - 0.5), 2.5, 1, boxstyle="round,pad=0.05",
                           facecolor=out_color, edgecolor='black', linewidth=1.5)
    ax.add_patch(obox)
    ax.text(8.25, y, out_label, ha='center', va='center', fontsize=10)
    ax.add_patch(FancyArrowPatch((5.2, y), (7.0, y), arrowstyle='->',
                                  mutation_scale=18, linewidth=1.5))

ax.set_title('Cách tạo Q, K, V từ embedding một token',
             fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

## D.3. Cách Attention "ghép cặp" Q với K

Sau khi mỗi token có Q và K riêng, làm sao biết token nào *liên quan* tới token nào?

**Câu trả lời: tích vô hướng (dot product).**

Với hai vector cùng chiều, tích vô hướng đo "mức độ thẳng hàng" giữa chúng:

- $\vec{Q}_i \cdot \vec{K}_j$ **lớn** → query của $i$ "khớp" với key của $j$ → token $i$ nên *chú ý* nhiều tới $j$.
- $\vec{Q}_i \cdot \vec{K}_j$ **nhỏ/âm** → không liên quan → bỏ qua.

Tính tất cả các cặp $(i, j)$, ta được lưới điểm số (score) kích thước $T \times T$ (với $T$ là độ dài chuỗi). Đây là **bản nháp** của Attention Pattern.

## D.4. Chuẩn hoá bằng Softmax

Score thô có thể là số dương lớn, âm lớn, hoặc gần 0. Để biến chúng thành **trọng số** dùng cho tổng có trọng số (weighted sum), ta cần:

- Mỗi trọng số nằm trong $[0, 1]$
- Tổng các trọng số trên một hàng = 1 (phân phối xác suất)

→ Áp dụng **Softmax** theo từng hàng. Trước softmax, chia thêm $\sqrt{d_k}$ để ổn định số học (sẽ giải thích kỹ ở phần E):

$$
\alpha_{ij} = \text{softmax}_j\!\left(\frac{\vec{Q}_i \cdot \vec{K}_j}{\sqrt{d_k}}\right)
$$

Lưới $\alpha_{ij}$ chính là **Attention Pattern** — "ánh xạ chú ý" giữa các cặp token.

## D.4b. 🧮 Ví dụ tính tay: attention trên câu 3 từ

Để "thấy" từng con số, xét câu **"đường ngọt quá"** với vector 2 chiều tự chọn (đã là Q, K, V — bỏ qua bước nhân ma trận để dễ theo dõi). Ta cập nhật từ **"đường"** (query ở hàng 0).

| Token | $\vec Q$ | $\vec K$ | $\vec V$ (ý nghĩa gợi ý) |
|---|---|---|---|
| đường | [1, 0] | [1, 0] | [1, 0] ("mơ hồ") |
| ngọt | [0, 1] | [2, 1] | [0, 3] ("vị ngọt") |
| quá | [0, 0] | [0, 0] | [0, 0] ("nhấn mạnh") |

**Bước 1 — điểm khớp** $s_j = \vec Q_{\text{đường}} \cdot \vec K_j / \sqrt{d_k}$ với $d_k = 2$:
$s = [1/\sqrt2,\ 2/\sqrt2,\ 0] \approx [0{,}71,\ 1{,}41,\ 0]$.

**Bước 2 — softmax:** $e^{0{,}71} \approx 2{,}03$, $e^{1{,}41} \approx 4{,}11$, $e^{0} = 1$; tổng $\approx 7{,}14$ → $\alpha \approx [0{,}28,\ 0{,}58,\ 0{,}14]$.

**Bước 3 — tổng có trọng số của Value:** $\Delta\vec E = 0{,}28\,[1,0] + 0{,}58\,[0,3] + 0{,}14\,[0,0] \approx [0{,}28,\ 1{,}73]$.

→ "đường" lấy **58%** thông tin từ "ngọt": vector mới nghiêng mạnh về chiều "vị ngọt". Cell dưới kiểm tra lại bằng code.

In [ ]:
Q = torch.tensor([[1., 0.], [0., 1.], [0., 0.]])
K = torch.tensor([[1., 0.], [2., 1.], [0., 0.]])
V = torch.tensor([[1., 0.], [0., 3.], [0., 0.]])
tokens_vi = ['đường', 'ngọt', 'quá']

scores = Q @ K.T / math.sqrt(2)
alpha = scores.softmax(dim=-1)
out = alpha @ V
torch.set_printoptions(precision=3, sci_mode=False)
print("Scores (hàng = query):\n", scores)
print("\nAttention α (mỗi hàng tổng = 1):\n", alpha)
print(f"\nCập nhật cho '{tokens_vi[0]}': {out[0].tolist()}  ← khớp với tính tay [0.28, 1.73]")

In [ ]:
# Minh hoạ Attention Pattern với một ví dụ có ngữ nghĩa (giả định)
# Câu: "A fluffy blue creature roamed the verdant forest"
tokens = ['A', 'fluffy', 'blue', 'creature', 'roamed', 'the', 'verdant', 'forest']
T = len(tokens)

# Tạo một attention pattern "lý tưởng" theo ngữ nghĩa
# (giả định mô hình đã học: tính từ -> danh từ kế cận)
ideal = np.eye(T) * 0.5  # mỗi token tự nhìn mình nhẹ
# fluffy(1), blue(2) -> creature(3)
ideal[3, 1] = 0.35
ideal[3, 2] = 0.35
ideal[3, 3] = 0.20
# verdant(6) -> forest(7)
ideal[7, 6] = 0.55
ideal[7, 7] = 0.30
ideal[7, 5] = 0.15
# Một số quan hệ khác
ideal[4, 3] = 0.4  # roamed nhìn creature
ideal[4, 4] = 0.3
ideal[4, 0] = 0.15
ideal[4, 1] = 0.15
# Chuẩn hoá từng hàng để tổng = 1
ideal = ideal / ideal.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(ideal, cmap='Blues', vmin=0, vmax=0.6)

for i in range(T):
    for j in range(T):
        v = ideal[i, j]
        if v > 0.01:
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    color='white' if v > 0.3 else 'black', fontsize=10)

ax.set_xticks(range(T)); ax.set_yticks(range(T))
ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(tokens, fontsize=10)
ax.set_xlabel('Key (token được nhìn vào)', fontsize=11)
ax.set_ylabel('Query (token đang đặt câu hỏi)', fontsize=11)
ax.set_title('Attention Pattern — minh hoạ "lý tưởng"\n'
             '(fluffy + blue → creature, verdant → forest)',
             fontsize=12)
plt.colorbar(im, ax=ax, label='Trọng số $\\alpha_{ij}$')
plt.tight_layout()
plt.show()

**Cách đọc heatmap:**

- *Hàng* `i` = token đặt câu hỏi (query).
- *Cột* `j` = token được "nhìn vào" (key).
- Ô $(i, j)$ = $\alpha_{ij}$: khi cập nhật token `i`, lấy bao nhiêu phần thông tin từ token `j`.
- **Mỗi hàng tổng = 1** (vì softmax theo hàng).

Trong heatmap trên (giả định mô hình đã học), bạn có thể thấy:

- Hàng "creature": chú ý mạnh tới "fluffy" và "blue" → hai tính từ đang tinh chỉnh ngữ nghĩa cho "creature".
- Hàng "forest": chú ý mạnh tới "verdant" → tính từ "verdant" đang đặc tả "forest".
- Hàng "roamed": chú ý tới "creature" (chủ ngữ) → động từ học mối quan hệ chủ-vị.

Tất nhiên, đây là pattern **lý tưởng do ta vẽ tay**. Pattern thật học được trong mô hình thường phức tạp hơn nhiều — và đó là vẻ đẹp của deep learning.

## D.5. Masking nhân quả (Causal Mask)

Trong các mô hình kiểu GPT (sinh văn bản từ trái sang phải), khi dự đoán token thứ $i$, mô hình **chỉ được nhìn vào các token $\le i$** — không được "nhìn trộm" token tương lai (nếu không sẽ rò rỉ đáp án trong huấn luyện).

Cách làm: **trước softmax**, gán các ô $\,Q_i \cdot K_j\,$ với $j > i$ bằng $-\infty$ (thực tế dùng `-1e9`). Sau softmax, $e^{-\infty} = 0$ → trọng số đúng bằng 0.

In [ ]:
# Trực quan hoá mặt nạ tam giác
def make_causal_mask(size):
    """Trả về mask shape (size, size); True = được phép nhìn."""
    return torch.tril(torch.ones(size, size, dtype=torch.bool))

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# Trái: không masking (encoder/BERT-style)
axes[0].imshow(np.ones((10, 10)), cmap='Greens', vmin=0, vmax=1.5)
for i in range(10):
    for j in range(10):
        axes[0].text(j, i, '✓', ha='center', va='center', color='darkgreen', fontsize=14)
axes[0].set_title('Không masking (Encoder / BERT)\nMỗi token nhìn được toàn bộ chuỗi',
                  fontsize=11)
axes[0].set_xlabel('Key position'); axes[0].set_ylabel('Query position')
axes[0].set_xticks(range(10)); axes[0].set_yticks(range(10))

# Phải: causal mask (decoder/GPT-style)
mask = make_causal_mask(10).numpy()
axes[1].imshow(mask, cmap='Greens', vmin=0, vmax=1.5)
for i in range(10):
    for j in range(10):
        if mask[i, j]:
            axes[1].text(j, i, '✓', ha='center', va='center', color='darkgreen', fontsize=14)
        else:
            axes[1].text(j, i, '✗', ha='center', va='center', color='red', fontsize=14)
axes[1].set_title('Causal masking (Decoder / GPT)\nToken sau KHÔNG ảnh hưởng token trước',
                  fontsize=11)
axes[1].set_xlabel('Key position'); axes[1].set_ylabel('Query position')
axes[1].set_xticks(range(10)); axes[1].set_yticks(range(10))

plt.tight_layout()
plt.show()

**Phân biệt 3 kiểu attention:**

| Kiểu | Mask | Ví dụ kiến trúc |
|------|------|----------------|
| **Encoder Self-Attention** | Không mask (chỉ mask padding) | BERT, T5 encoder |
| **Decoder Self-Attention** | Causal mask | GPT-2/3/4, decoder của Transformer gốc |
| **Cross-Attention** | Không mask về ngữ nghĩa (query từ decoder, K/V từ encoder) | Decoder của Transformer dịch máy |

**Vì sao kích thước attention pattern là $T^2$?**

Vì mỗi token cần 1 hàng (query) + 1 cột (key). Đây là lý do **context length** là giới hạn lớn của LLM: tăng $T$ gấp 2 → bộ nhớ + tính toán tăng gấp 4.

## D.6. Bước cuối: cập nhật embedding bằng Value

Sau khi có Attention Pattern $\alpha_{ij}$, **lượng cập nhật** cho embedding của token $i$ là:

$$
\Delta \vec{E}_i = \sum_{j=1}^{T} \alpha_{ij}\, \vec{V}_j
$$

— tức là tổng có trọng số của các Value vector, với trọng số là attention.

Embedding mới:

$$
\vec{E}'_i = \vec{E}_i + \Delta \vec{E}_i \quad \text{(qua residual connection)}
$$

Cách nhìn của 3Blue1Brown: thay vì một ma trận $W_V$ vuông, mỗi head có cặp **value-down** ($d_{\text{model}} \to d_k$) và **value-up** ($d_k \to d_{\text{model}}$). Trong khung "Transformer circuits" (Elhage et al., 2021), tích của hai ma trận này chính là **OV circuit** $W_O^h W_V^h$ (hạng $\le d_k$), còn $W_Q^{h\top} W_K^h$ là **QK circuit** quyết định *ai nhìn ai*.

**Tóm gọn toàn bộ quy trình** trong một công thức ma trận (Vaswani et al., 2017):

$$
\boxed{\;\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V\;}
$$

Đây gọi là **Scaled Dot-Product Attention**. Phần tiếp theo sẽ cài đặt nó.

---

# 🧮 E. Cài đặt Scaled Dot-Product Attention

> **⏱ Ý chính.** Ba phép toán: nhân ma trận $QK^\top$ → chia $\sqrt{d_k}$, che mask → softmax theo hàng → nhân với $V$. Chi phí bộ nhớ $O(T^2)$ vì phải giữ ma trận điểm $T \times T$.

Hàm dưới đây nhận:
- `query`: shape `(..., T_q, d_k)`
- `key`:   shape `(..., T_k, d_k)`
- `value`: shape `(..., T_k, d_v)`
- `mask`:  shape broadcast được tới `(..., T_q, T_k)` — quy ước theo Annotated Transformer: **1/True = được nhìn**, **0/False = bị che**. (Lưu ý: `torch.nn.MultiheadAttention` dùng quy ước *ngược lại* cho `attn_mask` kiểu bool — True là bị che. Nhầm quy ước là lỗi rất hay gặp.)

Trả về:
- `output`: shape `(..., T_q, d_v)` — embedding đã cập nhật.
- `attn`:   shape `(..., T_q, T_k)` — attention pattern (đã softmax).

In [ ]:
def attention(query, key, value, mask=None, dropout=None):
    """Scaled Dot-Product Attention (cài đặt theo Annotated Transformer).

    Quy ước mask: 1/True ở vị trí được phép nhìn; 0/False ở vị trí bị CHE.
    """
    d_k = query.size(-1)

    # 1) Tính score: Q @ K^T / sqrt(d_k)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

    # 2) Áp dụng mask (nếu có): điền -∞ ở vị trí bị che
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    # 3) Softmax theo chiều cuối (mỗi hàng = phân phối xác suất)
    p_attn = F.softmax(scores, dim=-1)

    # 4) Dropout (chỉ khi huấn luyện)
    if dropout is not None:
        p_attn = dropout(p_attn)

    # 5) Tổ hợp tuyến tính Value theo trọng số attention
    out = torch.matmul(p_attn, value)
    return out, p_attn


# Demo nhanh
B, T, d_k, d_v = 2, 5, 8, 8
Q = torch.randn(B, T, d_k)
K = torch.randn(B, T, d_k)
V = torch.randn(B, T, d_v)

out, attn = attention(Q, K, V)
print(f"Q shape: {tuple(Q.shape)}")
print(f"K shape: {tuple(K.shape)}")
print(f"V shape: {tuple(V.shape)}")
print(f"Output:  {tuple(out.shape)}  ← (B, T, d_v)")
print(f"Attn:    {tuple(attn.shape)} ← (B, T, T)")
print(f"\nKiểm tra mỗi hàng attention tổng = 1:")
print(f"  attn[0, 0].sum() = {attn[0, 0].sum().item():.4f}")

In [ ]:
# Đối chiếu với hàm tối ưu có sẵn của PyTorch (≥ 2.0): F.scaled_dot_product_attention
# Hàm này tự chọn backend FlashAttention / memory-efficient / math tuỳ phần cứng (xem Phần P.7).
q, k, v = torch.randn(2, 4, 6, 16), torch.randn(2, 4, 6, 16), torch.randn(2, 4, 6, 16)
causal = torch.tril(torch.ones(6, 6, dtype=torch.bool))          # True = được nhìn (với SDPA bool mask cũng vậy)

ours, _ = attention(q, k, v, mask=causal)
torch_out = F.scaled_dot_product_attention(q, k, v, attn_mask=causal)
torch_causal = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print("Khớp với SDPA (attn_mask):", torch.allclose(ours, torch_out, atol=1e-5))
print("Khớp với SDPA (is_causal) :", torch.allclose(ours, torch_causal, atol=1e-5))

## E.1. Vì sao chia $\sqrt{d_k}$?

Vaswani et al. (2017, chú thích 4) lập luận: nếu các thành phần của $\vec q$ và $\vec k$ độc lập, trung bình 0, phương sai 1, thì

$$\vec q \cdot \vec k = \sum_{i=1}^{d_k} q_i k_i \quad\Rightarrow\quad \mathbb E[\vec q\cdot\vec k] = 0,\qquad \operatorname{Var}(\vec q\cdot\vec k) = \sum_{i=1}^{d_k}\operatorname{Var}(q_i k_i) = d_k .$$

**Ví dụ số:** với $d_k = 64$, độ lệch chuẩn của score là $\sqrt{64} = 8$. Hai score chênh nhau cỡ 16 (hai độ lệch chuẩn) cho tỉ lệ softmax $e^{16} \approx 8{,}9$ triệu — softmax gần như one-hot, gradient ở các vị trí khác xấp xỉ 0 ("bão hoà"). Chia cho $\sqrt{d_k} = 8$ đưa độ lệch chuẩn về 1, chênh lệch chỉ còn 2 → tỉ lệ $e^2 \approx 7{,}4$: phân phối "mềm", gradient đi qua được.

Paper cũng ghi nhận: không chia $\sqrt{d_k}$ thì **additive attention** (Bahdanau) thắng dot-product khi $d_k$ lớn — chia $\sqrt{d_k}$ khắc phục điều đó.

In [ ]:
# Kiểm chứng thống kê: Var(q·k) ≈ d_k
for d in [16, 64, 256]:
    q_s, k_s = torch.randn(100_000, d), torch.randn(100_000, d)
    dots = (q_s * k_s).sum(-1)
    print(f"d_k = {d:3d}: Var(q·k) = {dots.var():7.1f}  (lý thuyết {d})   |  Var sau khi chia √d_k = {(dots / math.sqrt(d)).var():.3f}")

# Ảnh hưởng lên softmax: độ "nhọn" của phân phối
d = 256
s = torch.randn(10, d) @ torch.randn(d)
print(f"\nMax softmax KHÔNG chia √d_k: {s.softmax(0).max():.3f}   |   CÓ chia: {(s / math.sqrt(d)).softmax(0).max():.3f}")

In [ ]:
# Minh hoạ: ảnh hưởng của chia sqrt(d_k) lên phân phối softmax
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for ax, d_k_val, title in zip(axes, [16, 256], ['$d_k = 16$', '$d_k = 256$']):
    # Tạo 1000 mẫu Q, K ngẫu nhiên, tính score = Q.K
    Q = torch.randn(1000, d_k_val)
    K = torch.randn(1000, d_k_val)
    scores_unscaled = (Q * K).sum(dim=-1)
    scores_scaled = scores_unscaled / math.sqrt(d_k_val)

    ax.hist(scores_unscaled.numpy(), bins=50, alpha=0.5, label='Không chia $\\sqrt{d_k}$',
            color='red', density=True)
    ax.hist(scores_scaled.numpy(), bins=50, alpha=0.5, label='Chia $\\sqrt{d_k}$',
            color='blue', density=True)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Giá trị score'); ax.set_ylabel('Mật độ')
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Phân phối score trước/sau khi chia $\\sqrt{d_k}$',
             fontsize=13, weight='bold')
plt.tight_layout()
plt.show()
print("→ Khi d_k lớn, score không chia có biên độ rất lớn (đỏ rộng).")
print("→ Chia sqrt(d_k) giữ phương sai score ổn định ≈ 1 (xanh hẹp), giúp softmax không 'bão hoà'.")

---

# 🎭 F. Multi-Head Attention — chạy song song nhiều "đầu"

> **⏱ Ý chính.** Thay vì 1 attention lớn, chạy $h$ attention nhỏ ($d_k = d_{\text{model}}/h$) song song, mỗi head học một kiểu quan hệ, rồi ghép lại. Tổng chi phí tương đương một head lớn.

## F.1. Trực giác

**Một head** chỉ học được một loại "ai nhìn ai". Nhưng một câu có **rất nhiều** quan hệ:

- Tính từ → danh từ ("con mèo **đen**")
- Đại từ → tiền tố ("**nó**" → "con mèo")
- Động từ → tân ngữ ("đá **quả bóng**")
- Vị trí gần ("token kế trước")
- …

Giải pháp: chạy **nhiều head song song**, mỗi head có bộ $W_Q, W_K, W_V$ riêng, sau đó **ghép** (concatenate) đầu ra và chiếu qua $W^O$:

$$
\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)\, W^O,\qquad \text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V)
$$

**Ví dụ số (Transformer-base):** $d_{\text{model}} = 512$, $h = 8$ → mỗi head có $d_k = 64$. Chi phí tính $QK^\top$ của 8 head × 64 chiều = của 1 head × 512 chiều: đa đầu **không tốn thêm** phép nhân, chỉ đổi cách chia.

**Có cần nhiều head đến vậy không?** Voita et al. (2019) cắt bỏ 38/48 head encoder của một mô hình dịch Anh–Nga mà chỉ mất 0,15 BLEU; Michel, Levy & Neubig (2019) cũng thấy nhiều head có thể bỏ lúc suy luận. Một số ít head "chuyên biệt" gánh phần lớn công việc — gợi ý cho các kỹ thuật chia sẻ K/V (MQA/GQA, Phần P.3).

## F.3. Hàm tiện ích `clones`

Để tránh viết lặp lại, ta dùng helper sao chép sâu một module $N$ lần với **tham số độc lập**:

In [ ]:
def clones(module, N):
    """Tạo N bản deepcopy của module (mỗi bản có tham số riêng)."""
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [ ]:
class MultiHeadedAttention(nn.Module):
    """Multi-Head Attention theo paper gốc, gộp các head vào ma trận lớn."""

    def __init__(self, h: int, d_model: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % h == 0, "d_model phải chia hết cho h"

        self.d_k = d_model // h   # chiều mỗi head (paper: 512 / 8 = 64)
        self.h = h

        # 4 ma trận tuyến tính: W_Q, W_K, W_V, W_O (mỗi cái d_model × d_model)
        self.linears = clones(nn.Linear(d_model, d_model), 4)
        self.attn = None                      # lưu lại để có thể trực quan hoá
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        # query, key, value: (batch, seq_len, d_model)
        if mask is not None:
            # Mask shape (batch, 1, seq, seq) để broadcast cho tất cả head
            mask = mask.unsqueeze(1)

        nbatches = query.size(0)

        # 1) Chiếu Q, K, V qua linears[0..2], rồi tách thành (batch, h, seq, d_k)
        query, key, value = [
            l(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
            for l, x in zip(self.linears, (query, key, value))
        ]

        # 2) Áp dụng attention SONG SONG trên tất cả head
        x, self.attn = attention(query, key, value, mask=mask, dropout=self.dropout)

        # 3) Ghép các head: (batch, h, seq, d_k) -> (batch, seq, d_model)
        x = x.transpose(1, 2).contiguous().view(nbatches, -1, self.h * self.d_k)

        # 4) Chiếu cuối qua W_O
        return self.linears[-1](x)


# Demo
mha = MultiHeadedAttention(h=8, d_model=512, dropout=0.0)
x = torch.randn(2, 10, 512)   # batch=2, seq=10, d_model=512
out = mha(x, x, x)            # self-attention: Q=K=V=x

print(f"Input shape :  {tuple(x.shape)}")
print(f"Output shape:  {tuple(out.shape)}")
print(f"Attn shape  :  {tuple(mha.attn.shape)}  ← (batch, h, seq, seq)")
print(f"\nSố tham số của 1 khối MHA (d_model=512, h=8): "
      f"{sum(p.numel() for p in mha.parameters()):,}")

**Kiểm tra công thức số tham số:** 4 ma trận $d_{\text{model}} \times d_{\text{model}}$ cộng bias → $4(d^2 + d) = 4(512^2 + 512) = 1\,050\,624$ — khớp với con số in ra ở trên.

## F.4. Trực quan hoá các head song song

Mỗi head học một "góc nhìn" khác nhau. Hãy quan sát attention pattern của 4 head đầu tiên trên một câu giả lập (trọng số chưa qua huấn luyện nên các pattern còn ngẫu nhiên — nhưng vẫn cho thấy chúng *khác nhau*):

In [ ]:
# Demo với chuỗi 8 token
tokens = ['A', 'fluffy', 'blue', 'creature', 'roamed', 'the', 'verdant', 'forest']
T = len(tokens)

mha_demo = MultiHeadedAttention(h=4, d_model=64, dropout=0.0)
x = torch.randn(1, T, 64)
_ = mha_demo(x, x, x)
attn_all = mha_demo.attn[0]   # (h, T, T)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for h in range(4):
    ax = axes[h]
    im = ax.imshow(attn_all[h].detach().numpy(), cmap='Blues', vmin=0, vmax=0.5)
    ax.set_title(f'Head #{h+1}', fontsize=11)
    ax.set_xticks(range(T)); ax.set_yticks(range(T))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(tokens, fontsize=8)
    if h == 0:
        ax.set_ylabel('Query')
    ax.set_xlabel('Key')

plt.suptitle('Attention pattern của 4 head song song\n'
             '(trọng số chưa huấn luyện — chỉ minh hoạ rằng chúng KHÁC nhau)',
             fontsize=12, weight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Trong mô hình đã huấn luyện thật**, các nghiên cứu cho thấy nhiều head học các mẫu dễ nhận ra (Clark et al., 2019; Voita et al., 2019):

- Head "nhìn token kế trước" (sọc chéo ngay dưới đường chéo chính).
- Head "nhìn dấu câu / token phân tách" (cột `.`, `[SEP]` sáng) — thường đóng vai trò "không làm gì" (no-op).
- Head "cú pháp": động từ → tân ngữ, giới từ → danh từ.
- **Induction head** (Olsson et al., 2022): thấy mẫu `[A][B] … [A]` thì dự đoán `[B]` — cơ chế được gắn với khả năng *in-context learning* của LLM.

⚠️ **Cẩn trọng khi diễn giải:** trọng số attention **không tự động là lời giải thích** cho dự đoán. Jain & Wallace (2019) cho thấy có thể thay đổi mạnh attention mà dự đoán không đổi; Wiegreffe & Pinter (2019) phản biện rằng điều đó tuỳ vào định nghĩa "giải thích". Đây là chủ đề của **mechanistic interpretability** (Elhage et al., 2021).

---

# 🌐 G. Position-wise Feed-Forward Network (FFN)

> **⏱ Ý chính.** Attention *trộn* thông tin giữa các token; FFN *xử lý* từng token riêng lẻ bằng hai lớp Linear. FFN chiếm khoảng 2/3 số tham số và hoạt động như "bộ nhớ tri thức".

$$
\text{FFN}(x) = \max(0,\; x W_1 + b_1)\, W_2 + b_2
$$

Kích thước:
- Đầu vào & đầu ra: $d_{\text{model}}$ (512 trong paper gốc, 12 288 trong GPT-3).
- Tầng ẩn: $d_{\text{ff}} = 4\,d_{\text{model}}$ → 2048 trong paper gốc.

**Ví dụ số:** tham số FFN một tầng $= 2 \cdot d \cdot d_{ff} + d_{ff} + d = 2 \cdot 512 \cdot 2048 + 2048 + 512 = 2\,099\,712$ — gấp đôi MHA (1 050 624). Tỉ lệ $8d^2 : 4d^2$ này giữ nguyên ở GPT-3 (Phần L).

**Vai trò:** Geva et al. (2021) cho thấy mỗi FFN hoạt động như một **bộ nhớ key–value**: hàng của $W_1$ là "key" nhận diện mẫu đầu vào, hàng tương ứng của $W_2$ là "value" đẩy phân phối từ tiếp theo — nơi lưu các sự kiện kiểu "Hà Nội là thủ đô của Việt Nam".

**Phiên bản hiện đại:** LLM 2023–2026 thay ReLU bằng **SwiGLU** (Shazeer, 2020) — xem Phần P.10.

In [ ]:
class PositionwiseFeedForward(nn.Module):
    """FFN: hai lớp Linear, ReLU ở giữa, áp dụng độc lập tại mỗi vị trí."""

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)     # mở rộng d_model -> d_ff
        self.w_2 = nn.Linear(d_ff, d_model)     # thu lại d_ff -> d_model
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.w_2(self.dropout(F.relu(self.w_1(x))))


# Demo
ffn = PositionwiseFeedForward(d_model=512, d_ff=2048, dropout=0.0)
x = torch.randn(2, 10, 512)
print(f"Input  : {tuple(x.shape)}")
print(f"Output : {tuple(ffn(x).shape)}")
print(f"Số tham số: {sum(p.numel() for p in ffn.parameters()):,}")
print(f"  → 2 ma trận d_model × d_ff = 2 × 512 × 2048 + bias")

---

# 🧹 H. LayerNorm, RMSNorm & Residual Connection

> **⏱ Ý chính.** Residual ($x + f(x)$) cho gradient một "đường cao tốc" xuyên qua mạng sâu; LayerNorm giữ độ lớn activation ổn định. Đặt LayerNorm **trước** sub-layer (pre-LN) giúp huấn luyện dễ hơn — gần như mọi LLM hiện nay đều làm vậy.

## H.1. Layer Normalization

Batch Normalization không hợp với NLP (độ dài chuỗi, kích thước batch thay đổi). **Layer Normalization** (Ba, Kiros & Hinton, 2016) chuẩn hoá *trong cùng một token*, theo chiều feature:

$$
\text{LayerNorm}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta,\qquad \mu = \frac1d\sum_i x_i,\ \ \sigma^2 = \frac1d\sum_i (x_i-\mu)^2
$$

$\gamma, \beta$ là tham số học được.

**Ví dụ số:** $x = [1, 2, 3, 6]$ → $\mu = 3$, $\sigma^2 = (4+1+0+9)/4 = 3{,}5$, $\sigma \approx 1{,}87$ → $\hat x \approx [-1{,}07,\ -0{,}53,\ 0,\ 1{,}60]$.

⚠️ **Lưu ý cài đặt:** lớp `LayerNorm` bên dưới (theo Annotated Transformer) dùng `x.std()` — độ lệch chuẩn *không chệch* (chia $d-1$) — và cộng $\epsilon$ **vào** $\sigma$. `torch.nn.LayerNorm` dùng phương sai *chệch* (chia $d$) và cộng $\epsilon$ **trong** căn (mặc định $\epsilon = 10^{-5}$). Hai cách lệch nhau khoảng $\sqrt{d/(d-1)}$ — với $d = 512$ chỉ ~0,1%, nhưng **không trùng bit** (kiểm chứng bên dưới).

In [ ]:
class LayerNorm(nn.Module):
    """Cài đặt thủ công Layer Normalization (PyTorch có sẵn nn.LayerNorm)."""

    def __init__(self, features: int, eps: float = 1e-6):
        super().__init__()
        self.a_2 = nn.Parameter(torch.ones(features))    # gamma
        self.b_2 = nn.Parameter(torch.zeros(features))   # beta
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2


# Demo: kiểm tra LayerNorm cho ra trung bình ≈ 0, std ≈ 1 trên chiều cuối
ln = LayerNorm(features=64)
x = torch.randn(2, 5, 64) * 10 + 3   # cố ý lệch khỏi N(0,1)
y = ln(x)
print(f"Trước LayerNorm: mean={x.mean(-1).mean():.3f}, std={x.std(-1).mean():.3f}")
print(f"Sau LayerNorm  : mean={y.mean(-1).mean():.3f}, std={y.std(-1).mean():.3f}")

In [ ]:
# So sánh LayerNorm tự viết với nn.LayerNorm, và giới thiệu RMSNorm
x = torch.tensor([[1., 2., 3., 6.]])
print("Ví dụ tính tay  :", LayerNorm(4, eps=0.0)(x).detach().numpy().round(3), "(chia d-1 nên hơi khác)")
print("nn.LayerNorm    :", nn.LayerNorm(4, eps=0.0)(x).detach().numpy().round(3), "(khớp tính tay)")

x = torch.randn(3, 512)
ours, ref = LayerNorm(512)(x), nn.LayerNorm(512)(x)
print(f"\nd = 512: chênh lệch lớn nhất = {(ours - ref).abs().max():.5f}  → rất nhỏ nhưng khác 0")

class RMSNorm(nn.Module):
    """RMSNorm (Zhang & Sennrich, 2019): bỏ trừ trung bình và bias, chỉ chia cho căn trung bình bình phương.
    Dùng trong Llama, Qwen, DeepSeek, Gemma."""
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps, self.weight = eps, nn.Parameter(torch.ones(d))
    def forward(self, x):
        return self.weight * x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

y = RMSNorm(512)(x)
print(f"RMSNorm: RMS sau chuẩn hoá = {y.pow(2).mean(-1).sqrt().mean():.3f} (≈ 1); mean không bị ép về 0: {y.mean(-1).abs().mean():.3f}")

## H.2. Residual + Dropout + LayerNorm → SublayerConnection

Quanh mỗi sub-layer (Attention hoặc FFN), Transformer dùng **residual connection** (He et al., 2016): $\text{output} = x + \text{Sublayer}(x)$. Có hai cách đặt LayerNorm:

| Cách | Công thức | Ai dùng |
|---|---|---|
| **Post-LN** | $\text{LayerNorm}(x + \text{Sublayer}(x))$ | **Bài báo gốc** (Vaswani et al., 2017, §3.1), BERT |
| **Pre-LN** | $x + \text{Dropout}(\text{Sublayer}(\text{LayerNorm}(x)))$ | **Code Annotated Transformer** (ghi rõ: đặt norm trước "để code đơn giản"), GPT-2 trở đi, Llama, Qwen, DeepSeek |

Xiong et al. (2020) chứng minh: với post-LN, gradient ở các tầng gần đầu ra **rất lớn lúc khởi tạo**, nên bắt buộc phải có giai đoạn **warmup** learning rate (xem Noam schedule ở M.2); pre-LN có gradient "hiền" ngay từ đầu, có thể bỏ warmup và hội tụ nhanh hơn. Cái giá: pre-LN cần thêm một LayerNorm cuối cùng sau tầng cuối (chính là `self.norm` trong `Encoder`/`Decoder` ở phần I, J).

**Vai trò của 3 thành phần:**

| Thành phần | Vai trò |
|------------|---------|
| Residual ($+\,x$) | Gradient lan truyền không bị tắc nghẽn dù mạng sâu (như ResNet) |
| Dropout | Chống overfitting (Transformer-base: $P_{drop} = 0{,}1$) |
| LayerNorm | Giữ activation ở thang đo hợp lý |

In [ ]:
class SublayerConnection(nn.Module):
    """Bọc một sub-layer: x + Dropout(Sublayer(LayerNorm(x)))."""

    def __init__(self, size: int, dropout: float):
        super().__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        """sublayer là một callable; ví dụ: lambda x: mha(x, x, x, mask)."""
        return x + self.dropout(sublayer(self.norm(x)))

**Sơ đồ minh hoạ SublayerConnection (pre-norm):**

```
        x ────────────────────────────┐
        │                              │
   LayerNorm                          │
        │                              │
   Sublayer (Attn/FFN)                │
        │                              │
   Dropout                            │
        │                              │
        └──────────► (+) ◄──────────────┘
                     │
                     ▼
                  output
```

Đây là khối **lặp lại** trong mọi tầng của Encoder và Decoder.

---

# 📖 I. Encoder

Encoder là chồng $N$ tầng giống nhau. Mỗi tầng có **2 sub-layer**:

1. **Multi-Head Self-Attention** — mỗi token nhìn vào toàn bộ chuỗi nguồn.
2. **Position-wise FFN**.

Mỗi sub-layer được bọc trong `SublayerConnection`.

In [ ]:
class EncoderLayer(nn.Module):
    """Một tầng Encoder = Self-Attention + FFN, mỗi cái có residual+norm."""

    def __init__(self, size: int, self_attn, feed_forward, dropout: float):
        super().__init__()
        self.self_attn = self_attn          # MultiHeadedAttention
        self.feed_forward = feed_forward    # PositionwiseFeedForward
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size

    def forward(self, x, mask):
        # Sub-layer 1: self-attention (Q=K=V=x)
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
        # Sub-layer 2: feed-forward
        return self.sublayer[1](x, self.feed_forward)


class Encoder(nn.Module):
    """Encoder = chồng N EncoderLayer + LayerNorm cuối."""

    def __init__(self, layer, N: int):
        super().__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)


# Demo nhanh
attn = MultiHeadedAttention(h=8, d_model=512)
ffn = PositionwiseFeedForward(d_model=512, d_ff=2048)
encoder_layer = EncoderLayer(size=512, self_attn=copy.deepcopy(attn),
                              feed_forward=copy.deepcopy(ffn), dropout=0.1)
encoder = Encoder(encoder_layer, N=6)

x = torch.randn(2, 10, 512)
mask = torch.ones(2, 1, 10)  # mask đơn giản: không che gì
print(f"Encoder input  : {tuple(x.shape)}")
print(f"Encoder output : {tuple(encoder(x, mask).shape)}")
print(f"Tham số Encoder: {sum(p.numel() for p in encoder.parameters()):,}")

---

# 📝 J. Decoder & Cross-Attention

Decoder cũng có $N$ tầng, nhưng mỗi tầng có **3 sub-layer**:

1. **Masked Multi-Head Self-Attention** — query/key/value đều từ chuỗi đích, có **causal mask**.
2. **Cross-Attention** (còn gọi là "encoder-decoder attention") — query từ decoder, **key/value từ output encoder** (`memory`).
3. **Position-wise FFN**.

Cross-attention chính là cầu nối cho phép decoder "đọc" thông tin từ encoder.

## J.1. Hàm tạo Causal Mask

Cần một hàm tạo mặt nạ tam giác (cho self-attention của decoder):

In [ ]:
def subsequent_mask(size: int):
    """Mặt nạ shape (1, size, size); True = được nhìn, False = bị che.

    Vị trí (i, j) = True iff j <= i (decoder ở vị trí i chỉ thấy j ≤ i).
    """
    attn_shape = (1, size, size)
    mask = np.triu(np.ones(attn_shape), k=1).astype('uint8')
    return torch.from_numpy(mask) == 0


# Trực quan hoá
fig, ax = plt.subplots(figsize=(6, 6))
m = subsequent_mask(10)[0].numpy()
ax.imshow(m, cmap='Greens', interpolation='nearest')
for i in range(10):
    for j in range(10):
        ax.text(j, i, '✓' if m[i, j] else '✗',
                ha='center', va='center',
                color='darkgreen' if m[i, j] else 'darkred', fontsize=11)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel('Key position')
ax.set_ylabel('Query position')
ax.set_title('Subsequent Mask (kích thước 10)\n✓ = được nhìn, ✗ = bị che')
plt.tight_layout()
plt.show()

In [ ]:
class DecoderLayer(nn.Module):
    """DecoderLayer = Masked Self-Attn + Cross-Attn + FFN, mỗi cái có residual."""

    def __init__(self, size: int, self_attn, src_attn, feed_forward, dropout: float):
        super().__init__()
        self.size = size
        self.self_attn = self_attn          # masked self-attention trên chuỗi đích
        self.src_attn = src_attn            # cross-attention nhìn encoder
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 3)

    def forward(self, x, memory, src_mask, tgt_mask):
        """
        x       : embedding hiện tại của decoder (B, T_tgt, d_model)
        memory  : output encoder (B, T_src, d_model)
        src_mask: che padding của nguồn
        tgt_mask: che padding + token tương lai của đích
        """
        m = memory
        # 1) Masked self-attention: Q=K=V=x, dùng tgt_mask
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        # 2) Cross-attention: Q=x (decoder), K=V=memory (encoder), dùng src_mask
        x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
        # 3) Feed-forward
        return self.sublayer[2](x, self.feed_forward)


class Decoder(nn.Module):
    """Decoder = chồng N DecoderLayer + LayerNorm cuối."""

    def __init__(self, layer, N: int):
        super().__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)


# Demo
decoder_layer = DecoderLayer(size=512,
                              self_attn=copy.deepcopy(attn),
                              src_attn=copy.deepcopy(attn),
                              feed_forward=copy.deepcopy(ffn),
                              dropout=0.1)
decoder = Decoder(decoder_layer, N=6)

memory = torch.randn(2, 8, 512)   # đầu ra encoder cho chuỗi nguồn dài 8
tgt = torch.randn(2, 5, 512)      # decoder đang xử lý chuỗi đích dài 5
src_mask = torch.ones(2, 1, 8)
tgt_mask = subsequent_mask(5).expand(2, 5, 5)

out = decoder(tgt, memory, src_mask, tgt_mask)
print(f"Decoder input  : {tuple(tgt.shape)}")
print(f"Memory shape   : {tuple(memory.shape)}")
print(f"Decoder output : {tuple(out.shape)}")
print(f"Tham số Decoder: {sum(p.numel() for p in decoder.parameters()):,}")

---

# 🧩 K. Lắp ráp Full Transformer

## K.1. Lớp khung EncoderDecoder

Lớp này gói tất cả thành phần (encoder, decoder, embedding nguồn, embedding đích, generator) thành một mô hình hoàn chỉnh.

In [ ]:
class EncoderDecoder(nn.Module):
    """Khung Transformer hoàn chỉnh."""

    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed    # Embeddings + PositionalEncoding cho nguồn
        self.tgt_embed = tgt_embed    # Embeddings + PositionalEncoding cho đích
        self.generator = generator    # Linear + log_softmax đầu ra

    def forward(self, src, tgt, src_mask, tgt_mask):
        memory = self.encode(src, src_mask)
        return self.decode(memory, src_mask, tgt, tgt_mask)

    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)


class Generator(nn.Module):
    """Chiếu vector ẩn d_model -> vocab, rồi log_softmax."""

    def __init__(self, d_model: int, vocab: int):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab)

    def forward(self, x):
        return F.log_softmax(self.proj(x), dim=-1)

## K.2. Hàm `make_model` — "công thức" lắp ráp

Hàm dưới đây nhận vào các siêu tham số (giống paper gốc) và trả về một Transformer hoàn chỉnh sẵn sàng huấn luyện.

In [ ]:
def make_model(src_vocab: int, tgt_vocab: int,
                N: int = 6, d_model: int = 512, d_ff: int = 2048,
                h: int = 8, dropout: float = 0.1):
    """Lắp ráp một Transformer Encoder-Decoder hoàn chỉnh.

    Siêu tham số mặc định theo paper gốc 'Attention Is All You Need'.
    """
    c = copy.deepcopy
    attn_proto = MultiHeadedAttention(h, d_model)
    ff_proto = PositionwiseFeedForward(d_model, d_ff, dropout)
    pos_proto = PositionalEncoding(d_model, dropout)

    model = EncoderDecoder(
        encoder=Encoder(EncoderLayer(d_model, c(attn_proto), c(ff_proto), dropout), N),
        decoder=Decoder(DecoderLayer(d_model,
                                       c(attn_proto), c(attn_proto), c(ff_proto),
                                       dropout), N),
        src_embed=nn.Sequential(Embeddings(d_model, src_vocab), c(pos_proto)),
        tgt_embed=nn.Sequential(Embeddings(d_model, tgt_vocab), c(pos_proto)),
        generator=Generator(d_model, tgt_vocab),
    )

    # Khởi tạo Xavier cho các tham số (giúp huấn luyện ổn định)
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    return model


# Tạo thử mô hình NHỎ (để nhìn được cấu trúc)
small_model = make_model(src_vocab=100, tgt_vocab=100, N=2, d_model=64, d_ff=256, h=4)
print("=== Cấu trúc Transformer nhỏ (N=2, d_model=64, h=4) ===")
print(small_model)
print(f"\nTổng số tham số: {sum(p.numel() for p in small_model.parameters()):,}")

**Lưu ý quan trọng — `copy.deepcopy`:**

Trong `make_model`, ta dùng `c(attn_proto)` mỗi lần truyền `MultiHeadedAttention` vào các tầng khác nhau. Điều này đảm bảo:

- Mỗi EncoderLayer có **bộ trọng số attention riêng**.
- Self-attention và cross-attention của Decoder cũng **độc lập** với nhau.

Nếu không deepcopy, tất cả sẽ **share trọng số** — không đúng với paper.

---

# 🔢 L. Đếm tham số: từ Transformer gốc đến GPT-3

> **⏱ Ý chính.** Mỗi tầng ≈ $4d^2$ (attention) + $8d^2$ (FFN) = $12d^2$ tham số. Nhân với số tầng là ra gần đúng kích thước mô hình.

## L.0. Kiểm chứng: Transformer-base có 65M tham số?

In [ ]:
# Transformer-base với vocab BPE dùng chung 37 000 (Vaswani et al., 2017)
V_BPE = 37_000
base = make_model(V_BPE, V_BPE, N=6, d_model=512, d_ff=2048, h=8)
n_untied = sum(p.numel() for p in base.parameters())
emb = V_BPE * 512
print(f"make_model (3 ma trận embedding/đầu ra RIÊNG): {n_untied/1e6:.1f}M tham số")
print(f"Nếu dùng chung (weight tying) 3 ma trận      : {(n_untied - 2 * emb)/1e6:.1f}M tham số")
print("→ Paper báo 65M: gần với bản có weight tying. make_model không tying nên lớn hơn ~38M.")
del base

---

# 🔢 L. Đếm tham số: từ Transformer gốc đến GPT-3

## L.1. Đếm cho 1 head attention

Trong **GPT-3** (175 tỉ tham số):
- $d_{\text{model}} = 12{,}288$
- $d_k = d_v = 128$ (do $h = 96$, nên $d_{\text{model}} / h = 128$)
- Số tầng $L = 96$

Cấu hình này khớp Bảng 2.1 của Brown et al. (2020): 96 tầng, $d_{\text{model}} = 12\,288$, 96 head, $d_{\text{head}} = 128$, ngữ cảnh 2048 token.

Cách 3Blue1Brown (Sanderson, 2024) trình bày Q, K, V, O dưới góc nhìn từng head:

| Ma trận | Kích thước | Số tham số |
|---------|------------|-----------|
| $W_Q^{(\text{head})}$ | $d_k \times d_{\text{model}} = 128 \times 12{,}288$ | 1,572,864 |
| $W_K^{(\text{head})}$ | $128 \times 12{,}288$ | 1,572,864 |
| $W_{V\downarrow}^{(\text{head})}$ | $128 \times 12{,}288$ | 1,572,864 |
| $W_{V\uparrow}^{(\text{head})}$ | $12{,}288 \times 128$ | 1,572,864 |
| **Tổng** | | **~6.3M** |

## L.2. Mẹo "low-rank" trên Value

Nếu để $W_V$ là ma trận vuông $d_{\text{model}} \times d_{\text{model}}$, mỗi head sẽ tốn:

$$
12{,}288 \times 12{,}288 \approx 151\text{ triệu tham số}
$$

→ Gấp **96 lần** $W_Q$ hoặc $W_K$! Quá lãng phí.

**Mẹo trong thực tế:** *phân tách* (factorize) $W_V$ thành hai ma trận nhỏ:

$$
W_V \;\approx\; W_{V\uparrow}\; W_{V\downarrow}
$$

- $W_{V\downarrow}$: chiếu $d_{\text{model}} \to d_k$ (nén xuống).
- $W_{V\uparrow}$: chiếu $d_k \to d_{\text{model}}$ (mở rộng lại).

Tổng tham số chỉ còn $2 \times d_{\text{model}} \times d_k = $ cùng bậc với $W_Q, W_K$. Đây là **low-rank transformation** — một kỹ thuật phổ biến trong deep learning.

## L.3. Thực tế trong code: gộp các head

> 📝 **Ghi chú:** cặp $W_{V\uparrow} W_{V\downarrow}$ của 3B1B chính là cặp $W_O^h W_V^h$ trong cài đặt chuẩn — số tham số như nhau ($2 \cdot d_{\text{model}} \cdot d_k$ mỗi head, tức $2d^2$ mỗi tầng), chỉ khác cách *diễn giải*: "OV circuit" hạng thấp (Elhage et al., 2021).

In [ ]:
# So sánh số tham số kiểu vuông vs kiểu phân tách
d_model_gpt3 = 12_288
d_k_gpt3 = 128
h_gpt3 = 96      # số head per block
L_gpt3 = 96      # số block

# Mỗi head
p_Q_head = d_model_gpt3 * d_k_gpt3
p_K_head = d_model_gpt3 * d_k_gpt3
p_V_factor_head = 2 * d_model_gpt3 * d_k_gpt3   # V↓ + V↑
p_V_naive = d_model_gpt3 * d_model_gpt3         # nếu V vuông

print(f"{'Cấu hình GPT-3:':<40} d_model={d_model_gpt3:,}, d_k={d_k_gpt3}, h={h_gpt3}, L={L_gpt3}")
print()
print(f"{'Một W_Q (hoặc W_K) per head':<40}: {p_Q_head:>15,} tham số")
print(f"{'W_V kiểu vuông (per head)':<40}: {p_V_naive:>15,} tham số  ⚠️ phí")
print(f"{'W_V phân tách (V↓ + V↑) per head':<40}: {p_V_factor_head:>15,} tham số  ✅ tiết kiệm")
print(f"{'Tổng (Q + K + V↓ + V↑) per head':<40}: {p_Q_head + p_K_head + p_V_factor_head:>15,}")
print()

# Mỗi block (96 head)
per_block = h_gpt3 * (p_Q_head + p_K_head + p_V_factor_head)
print(f"{'Tham số attention mỗi block (96 head)':<40}: {per_block:>15,}  ≈ {per_block/1e6:.0f}M")

# Toàn mô hình (96 block)
total_attn = L_gpt3 * per_block
print(f"{'Tham số attention toàn GPT-3':<40}: {total_attn:>15,}  ≈ {total_attn/1e9:.1f}B")
print()
print(f"💡 Toàn bộ GPT-3 có 175B tham số → attention chiếm khoảng "
      f"{total_attn/175e9*100:.0f}%; phần còn lại chủ yếu là FFN.")

## L.4. Trực quan hoá: phân bổ tham số trong Transformer

In [ ]:
# Phân bổ tham số ước lượng cho 1 layer GPT-3
d_model = d_model_gpt3
d_ff = 4 * d_model      # quy ước phổ biến: d_ff = 4 * d_model

attn_params_per_layer = h_gpt3 * (4 * d_model * d_k_gpt3)
# 4 vì Q, K, V↓, V↑. Trong code thực tế: 4 ma trận d_model × d_model = 4 * d_model^2 (cùng kết quả)
ffn_params_per_layer = 2 * d_model * d_ff   # W_1 (d_model x d_ff) + W_2 (d_ff x d_model)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Trái: per layer
labels = ['Multi-Head\nAttention', 'Feed-Forward\n(MLP)']
sizes = [attn_params_per_layer, ffn_params_per_layer]
colors = ['#FFB6C1', '#90EE90']
axes[0].pie(sizes, labels=labels, autopct=lambda p: f'{p:.0f}%\n({p/100*sum(sizes)/1e6:.0f}M)',
            colors=colors, startangle=90, textprops={'fontsize': 11})
axes[0].set_title(f'Phân bổ tham số trong 1 layer GPT-3\n'
                   f'(tổng ≈ {sum(sizes)/1e6:.0f}M tham số/layer)',
                   fontsize=12)

# Phải: toàn mô hình
total_attn_all = L_gpt3 * attn_params_per_layer
total_ffn_all = L_gpt3 * ffn_params_per_layer
embed_params = 50_257 * d_model      # ước tính embedding (vocab~50k)
other = 175e9 - total_attn_all - total_ffn_all - embed_params

labels2 = ['Attention (Q,K,V,O)', 'Feed-Forward', 'Embeddings', 'Khác (LayerNorm, bias…)']
sizes2 = [total_attn_all, total_ffn_all, embed_params, max(other, 0)]
colors2 = ['#FFB6C1', '#90EE90', '#FFE4B5', '#D3D3D3']
axes[1].pie(sizes2, labels=labels2,
            autopct=lambda p: f'{p:.1f}%\n({p/100*sum(sizes2)/1e9:.1f}B)',
            colors=colors2, startangle=90, textprops={'fontsize': 10})
axes[1].set_title(f'Phân bổ tham số toàn GPT-3\n(tổng ≈ 175B)', fontsize=12)

plt.tight_layout()
plt.show()
print("→ FFN thực ra 'chiếm' nhiều tham số HƠN attention — dù mọi người hay nhắc đến attention!")

**Kiểm tra cuối (tính toán):** attention $4d^2 \times 96 \approx 57{,}98$B; FFN $8d^2 \times 96 \approx 115{,}96$B; tổng các khối $12\,L\,d^2 \approx 173{,}9$B. Cộng embedding token ($50\,257 \times 12\,288 \approx 0{,}62$B, với 50 257 là kích thước tokenizer GPT-2) và embedding vị trí ($2048 \times 12\,288 \approx 0{,}025$B) → **≈ 174,6B**, khớp con số 175B công bố.

→ Attention chiếm khoảng **1/3**, FFN khoảng **2/3** số tham số. Nhưng khi *suy luận* với ngữ cảnh dài, thứ tốn bộ nhớ nhất lại là **KV-cache** của attention — chủ đề của Phần P.

---

# 🎓 M. Huấn luyện Transformer

## M.1. Đóng gói Batch + Mask

Khi huấn luyện, ta nhóm nhiều cặp (src, tgt) thành một batch. Vì các chuỗi có độ dài khác nhau, ta **pad** chúng và tạo mask để bỏ qua các vị trí padding.

In [ ]:
class Batch:
    """Một batch dữ liệu kèm các mask cần thiết."""

    def __init__(self, src, trg=None, pad: int = 0):
        self.src = src
        # src_mask: 1 ở vị trí có token thật, 0 ở padding
        # shape (B, 1, T_src) — sẽ broadcast trong attention
        self.src_mask = (src != pad).unsqueeze(-2)

        if trg is not None:
            # Cơ chế "shifted right":
            #   trg đầu vào decoder = bỏ token cuối
            #   trg_y (target để dự đoán) = bỏ token đầu
            # → ở vị trí i decoder phải dự đoán token (i+1) trong chuỗi gốc
            self.trg = trg[:, :-1]
            self.trg_y = trg[:, 1:]
            self.trg_mask = self.make_std_mask(self.trg, pad)
            self.ntokens = (self.trg_y != pad).data.sum()

    @staticmethod
    def make_std_mask(tgt, pad):
        """Kết hợp padding mask + subsequent (causal) mask."""
        tgt_mask = (tgt != pad).unsqueeze(-2)
        tgt_mask = tgt_mask & subsequent_mask(tgt.size(-1)).type_as(tgt_mask.data)
        return tgt_mask

**Giải thích cơ chế "shifted right" (teacher forcing):**

Giả sử chuỗi đích đầy đủ là `[<s>, Tôi, yêu, em, </s>]`:

| Bước | Decoder thấy (`trg`) | Phải dự đoán (`trg_y`) |
|------|---------------------|----------------------|
| pos 0 | `<s>` | `Tôi` |
| pos 1 | `<s>, Tôi` | `yêu` |
| pos 2 | `<s>, Tôi, yêu` | `em` |
| pos 3 | `<s>, Tôi, yêu, em` | `</s>` |

Nhờ causal mask, tại mỗi vị trí decoder *chỉ nhìn* các token đứng trước. Mô hình học tất cả các bước đồng thời (parallel), khác với RNN phải chạy tuần tự.

## M.2. Optimizer + Learning Rate Schedule ("Noam")

Paper gốc dùng **Adam** (Kingma & Ba, 2015) với $\beta_1 = 0{,}9$, $\beta_2 = 0{,}98$, $\epsilon = 10^{-9}$ (khác mặc định 0,999 và $10^{-8}$) và lịch trình:

$$
lr = d_{\text{model}}^{-0.5} \cdot \min\!\left(\text{step}^{-0.5},\; \text{step} \cdot \text{warmup}^{-1.5}\right)
$$

- **Warmup** (4000 bước đầu): lr tăng tuyến tính từ 0 → đỉnh.
- Sau đó: lr giảm theo $1/\sqrt{\text{step}}$.

**Ví dụ số:** đỉnh đạt tại step = warmup: $lr_{\max} = 512^{-0.5} \cdot 4000^{-0.5} = 0{,}0442 \times 0{,}0158 \approx 7{,}0 \times 10^{-4}$.

Tại sao cần warmup? Với **post-LN**, gradient lúc khởi tạo rất lớn ở các tầng cuối (Xiong et al., 2020); lr cao ngay từ đầu sẽ làm huấn luyện phân kỳ. (Tên "Noam" là cách gọi của cộng đồng Tensor2Tensor/OpenNMT theo tên Noam Shazeer, không xuất hiện trong bài báo.)

In [ ]:
class NoamOpt:
    """Wrapper quanh optimizer; tự điều chỉnh learning rate theo lịch trình Noam."""

    def __init__(self, model_size: int, factor: float, warmup: int, optimizer):
        self.optimizer = optimizer
        self._step = 0
        self.warmup = warmup
        self.factor = factor
        self.model_size = model_size
        self._rate = 0

    def step(self):
        self._step += 1
        rate = self.rate()
        for p in self.optimizer.param_groups:
            p['lr'] = rate
        self._rate = rate
        self.optimizer.step()

    def rate(self, step: int = None):
        if step is None:
            step = self._step
        return self.factor * (
            self.model_size ** (-0.5)
            * min(step ** (-0.5), step * self.warmup ** (-1.5))
        )


# Trực quan hoá lịch trình
opts = [
    ('d_model=512, warmup=4000', NoamOpt(512, 1, 4000, None)),
    ('d_model=512, warmup=8000', NoamOpt(512, 1, 8000, None)),
    ('d_model=256, warmup=4000', NoamOpt(256, 1, 4000, None)),
]

plt.figure(figsize=(10, 5))
steps = np.arange(1, 20000)
for label, opt in opts:
    plt.plot(steps, [opt.rate(int(s)) for s in steps], label=label, linewidth=2)
plt.xlabel('Bước (step)'); plt.ylabel('Learning rate')
plt.title('Lịch trình learning rate Noam\n'
          '(tăng tuyến tính → giảm theo $1/\\sqrt{step}$)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## M.3. Label Smoothing — chống "quá tự tin"

Với nhãn one-hot, cross-entropy ép mô hình xuất xác suất $\approx 1$ cho token đúng → dễ *quá tự tin*. **Label Smoothing** (Szegedy et al., 2016) "làm mượt" nhãn: thay $[0, 0, 1, 0, 0]$ bằng $[\epsilon/(V-1), \dots, 1-\epsilon, \dots]$ với $\epsilon$ nhỏ (paper: $\epsilon_{ls} = 0{,}1$).

Theo Vaswani et al. (2017): label smoothing **làm perplexity xấu đi** (mô hình "kém chắc chắn" hơn) nhưng **tăng accuracy và BLEU**. Müller, Kornblith & Hinton (2019) giải thích nó cải thiện *calibration* nhưng làm **knowledge distillation kém hiệu quả** hơn nhiều — cần lưu ý nếu định dùng mô hình làm "thầy".

⚠️ **Lưu ý cài đặt:** lớp `LabelSmoothing` dưới đây (theo Annotated Transformer) dùng `KLDivLoss(reduction='sum')` và chia đều $\epsilon$ cho $V - 2$ lớp (trừ token đúng và padding). Giá trị loss vì thế **khác** `nn.CrossEntropyLoss(label_smoothing=0.1)` một hằng số (entropy của phân phối mục tiêu) và cách chia $\epsilon$ — gradient thì tương đương.

In [ ]:
class LabelSmoothing(nn.Module):
    """Label smoothing dùng KL divergence loss."""

    def __init__(self, size: int, padding_idx: int, smoothing: float = 0.0):
        super().__init__()
        self.criterion = nn.KLDivLoss(reduction='sum')
        self.padding_idx = padding_idx
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.size = size
        self.true_dist = None

    def forward(self, x, target):
        """
        x: log-probabilities (N, vocab)
        target: chỉ số token đúng, shape (N,)
        """
        assert x.size(1) == self.size
        true_dist = x.data.clone()

        # Chia đều smoothing cho (vocab - 2): trừ vị trí token đúng và padding
        true_dist.fill_(self.smoothing / (self.size - 2))

        # Gán confidence cho vị trí token đúng
        true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)

        # Đặt 0 cho cột padding
        true_dist[:, self.padding_idx] = 0

        # Zero-out toàn bộ hàng nếu target là padding
        mask = torch.nonzero(target.data == self.padding_idx)
        if mask.dim() > 0:
            true_dist.index_fill_(0, mask.squeeze(), 0.0)

        self.true_dist = true_dist
        return self.criterion(x, true_dist.detach())


# Trực quan hoá phân phối "true" với label smoothing
crit = LabelSmoothing(size=5, padding_idx=0, smoothing=0.1)
predict = torch.FloatTensor([[0, 0.2, 0.7, 0.1, 0]]).log()   # log-probs (giả lập)
target = torch.LongTensor([2])    # token đúng = chỉ số 2
crit(predict, target)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(5), crit.true_dist[0].numpy(), color=['gray', 'steelblue', 'tomato', 'steelblue', 'steelblue'])
ax.set_xticks(range(5))
ax.set_xticklabels(['pad (0)', 'token 1', 'token 2\n(ĐÚNG)', 'token 3', 'token 4'])
ax.set_ylabel('Xác suất "true" sau smoothing')
ax.set_title('Phân phối nhãn sau Label Smoothing (smoothing=0.1)\n'
             'Token đúng: 0.9; mỗi token sai: 0.1/(V-2) ≈ 0.033')
for i, v in enumerate(crit.true_dist[0].numpy()):
    if v > 0:
        ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## M.4. Hàm `run_epoch` — vòng lặp huấn luyện cơ bản

In [ ]:
def run_epoch(data_iter, model, loss_compute, verbose=True):
    """Chạy 1 epoch; trả về loss/token trung bình."""
    start = time.time()
    total_tokens = 0
    total_loss = 0
    tokens = 0

    for i, batch in enumerate(data_iter):
        out = model.forward(batch.src, batch.trg, batch.src_mask, batch.trg_mask)
        loss = loss_compute(out, batch.trg_y, batch.ntokens)
        total_loss += loss
        total_tokens += batch.ntokens
        tokens += batch.ntokens

        if verbose and i % 50 == 1:
            elapsed = time.time() - start
            print(f"  Step {i:4d} | Loss/token: {loss / batch.ntokens:.4f} "
                  f"| Tokens/sec: {tokens / elapsed:7.1f}")
            start = time.time()
            tokens = 0

    return total_loss / total_tokens


class SimpleLossCompute:
    """Tính loss + backward + cập nhật optimizer (gói gọn cho run_epoch)."""

    def __init__(self, generator, criterion, opt=None):
        self.generator = generator
        self.criterion = criterion
        self.opt = opt

    def __call__(self, x, y, norm):
        x = self.generator(x)
        loss = self.criterion(
            x.contiguous().view(-1, x.size(-1)),
            y.contiguous().view(-1)
        ) / norm
        # Chỉ backward + cập nhật khi đang ở chế độ huấn luyện (có optimizer)
        if self.opt is not None:
            loss.backward()
            self.opt.step()
            self.opt.optimizer.zero_grad()
        return loss.data.item() * norm

---

# 🎯 N. Demo end-to-end: Bài toán Copy

Để kiểm chứng kiến trúc hoạt động, ta tạo bài toán đồ chơi: **dạy mô hình sao chép chính xác chuỗi đầu vào**.

- Từ điển: số nguyên 1–10 (token 0 = padding).
- Mỗi mẫu: chuỗi 10 số ngẫu nhiên; `src = tgt`.

Nếu mô hình hoạt động đúng, sau vài epoch nó sẽ học sao chép y nguyên.

In [ ]:
def data_gen(V, batch, nbatches):
    """Sinh batch dữ liệu cho bài toán copy."""
    for _ in range(nbatches):
        data = torch.from_numpy(np.random.randint(1, V, size=(batch, 10)))
        data[:, 0] = 1   # cố định token đầu = 1 (mô phỏng <start>)
        src = data.clone().long().requires_grad_(False)
        tgt = data.clone().long().requires_grad_(False)
        yield Batch(src, tgt, pad=0)


# Cấu hình
V = 11   # vocab: 0 (pad) + 1..10
criterion = LabelSmoothing(size=V, padding_idx=0, smoothing=0.0)
model = make_model(V, V, N=2, d_model=64, d_ff=128, h=4)
model_opt = NoamOpt(
    model.src_embed[0].d_model, factor=1, warmup=400,
    optimizer=torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
)

print("Bắt đầu huấn luyện…\n")
N_EPOCHS = 20   # bản gốc: 10 — chưa đủ để hội tụ với mô hình nhỏ này
train_losses, val_losses = [], []

for epoch in range(N_EPOCHS):
    print(f"=== Epoch {epoch + 1}/{N_EPOCHS} ===")

    model.train()
    train_loss = run_epoch(data_gen(V, 30, 20), model,
                            SimpleLossCompute(model.generator, criterion, model_opt),
                            verbose=False)

    model.eval()
    with torch.no_grad():
        val_loss = run_epoch(data_gen(V, 30, 5), model,
                              SimpleLossCompute(model.generator, criterion, None),
                              verbose=False)

    print(f"  Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")
    train_losses.append(train_loss.item() if hasattr(train_loss, 'item') else float(train_loss))
    val_losses.append(val_loss.item() if hasattr(val_loss, 'item') else float(val_loss))

print("\n✓ Huấn luyện xong!")

In [ ]:
# Vẽ đường loss
plt.figure(figsize=(9, 4.5))
plt.plot(train_losses, marker='o', label='Train loss/token')
plt.plot(val_losses, marker='s', label='Val loss/token')
plt.xlabel('Epoch'); plt.ylabel('Loss / token')
plt.title('Tiến trình huấn luyện trên bài toán Copy')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## N.1. Greedy Decoding

Sau khi huấn luyện, ta sinh chuỗi bằng **greedy decoding**: mỗi bước chọn token có log-prob cao nhất. (Paper gốc dùng **beam search** với beam = 4 và hệ số phạt độ dài $\alpha = 0{,}6$ — giữ 4 chuỗi tốt nhất song song thay vì 1; với bài toán copy, greedy là đủ.)

Lưu ý hiệu năng: hàm dưới đây chạy lại decoder trên **toàn bộ** tiền tố ở mỗi bước — độ phức tạp $O(T^2)$ lần tính. LLM thực tế dùng **KV-cache** để chỉ tính cho token mới (Phần P.2).

In [ ]:
def greedy_decode(model, src, src_mask, max_len: int, start_symbol: int):
    """Sinh chuỗi từng token, mỗi bước chọn argmax."""
    memory = model.encode(src, src_mask)
    ys = torch.ones(1, 1).fill_(start_symbol).type_as(src.data)

    for _ in range(max_len - 1):
        out = model.decode(
            memory, src_mask,
            ys,
            subsequent_mask(ys.size(1)).type_as(src.data)
        )
        prob = model.generator(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.data[0]
        ys = torch.cat([ys, torch.ones(1, 1).type_as(src.data).fill_(next_word)], dim=1)

    return ys


# Kiểm thử với nhiều chuỗi khác nhau
model.eval()
print("Test bài toán copy (mô hình tự sao chép input):\n")

test_cases = [
    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    [1, 10, 9, 8, 7, 6, 5, 4, 3, 2],
    [1, 5, 5, 5, 5, 5, 5, 5, 5, 5],
    [1, 3, 7, 2, 9, 4, 6, 8, 10, 5],
]

correct = 0
for src_list in test_cases:
    src = torch.LongTensor([src_list])
    src_mask = torch.ones(1, 1, 10)
    pred = greedy_decode(model, src, src_mask, max_len=10, start_symbol=1).tolist()[0]
    match = '✓' if pred == src_list else '✗'
    if pred == src_list:
        correct += 1
    print(f"  Input : {src_list}")
    print(f"  Output: {pred}  {match}\n")

print(f"Độ chính xác: {correct}/{len(test_cases)} chuỗi đúng hoàn toàn.")

## N.2. Trực quan hoá Attention sau huấn luyện

Hãy xem mô hình đã học pattern attention như thế nào — đặc biệt là **cross-attention** trong decoder (nhìn từ token đích về token nguồn). Với bài toán copy lý tưởng, ta kỳ vọng pattern này gần với **ma trận đơn vị**.

In [ ]:
# Lấy 1 mẫu, đẩy qua mô hình, lưu attention pattern
model.eval()
test_seq = [1, 3, 7, 2, 9, 4, 6, 8, 10, 5]
src = torch.LongTensor([test_seq])
src_mask = torch.ones(1, 1, 10)
tgt = torch.LongTensor([test_seq[:-1]])
tgt_mask = subsequent_mask(tgt.size(-1)).expand(1, tgt.size(-1), tgt.size(-1))

with torch.no_grad():
    _ = model(src, tgt, src_mask, tgt_mask)

# Lấy attention pattern của cross-attention ở tầng cuối decoder
cross_attn = model.decoder.layers[-1].src_attn.attn[0]    # (h, T_tgt, T_src)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for h in range(4):
    a = cross_attn[h].numpy()
    im = axes[h].imshow(a, cmap='Blues', vmin=0, vmax=a.max())
    axes[h].set_title(f'Cross-Attn Head #{h+1}')
    axes[h].set_xticks(range(10)); axes[h].set_yticks(range(9))
    axes[h].set_xticklabels(test_seq, fontsize=9)
    axes[h].set_yticklabels(test_seq[:-1], fontsize=9)
    axes[h].set_xlabel('Token nguồn (Key)')
    if h == 0:
        axes[h].set_ylabel('Token đích (Query)')

plt.suptitle('Cross-Attention sau huấn luyện trên bài toán Copy\n'
             '(kỳ vọng: token đích i ↔ token nguồn (i+1))',
             fontsize=12, weight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Nhận xét:** Nếu mô hình đã học tốt, bạn sẽ thấy một số head có **đường chéo sáng** — nghĩa là khi sinh token thứ $i$ ở đích, decoder đang chú ý vào token tương ứng (hoặc gần đó) ở nguồn. Pattern chính xác phụ thuộc vào số epoch và cách khởi tạo, nhưng nguyên lý là vậy.

Nếu vẫn còn chuỗi sai, hãy **tăng `N_EPOCHS`** (30–50) — mô hình nhỏ và lịch Noam với warmup ngắn cần nhiều bước hơn để hội tụ.

---

# 🚀 P. Attention hiện đại 2019–2026: bài toán mới là *bộ nhớ khi suy luận*

> **⏱ Ý chính.** Năm 2017, câu hỏi là "dịch có tốt hơn RNN không". Năm 2023–2026, câu hỏi là "**mỗi token sinh ra tốn bao nhiêu bộ nhớ và thời gian, với ngữ cảnh 128K–1M token**". Mọi cải tiến dưới đây trả lời câu hỏi đó ở một chỗ khác nhau:
>
> | Nút cổ chai | Cải tiến | Mục |
> |---|---|---|
> | KV-cache lớn | MQA → **GQA** → **MLA** | P.1–P.4 |
> | Vị trí không ngoại suy | **RoPE**, **ALiBi**, YaRN | P.5–P.6 |
> | Ma trận $T\times T$ không vừa bộ nhớ GPU | **FlashAttention** 1–4 | P.7 |
> | $O(T^2)$ với ngữ cảnh rất dài | **Sliding-window**, sparse học được (NSA/DSA), **linear attention / SSM** | P.8–P.9 |
> | Huấn luyện mất ổn định ở quy mô lớn | Pre-**RMSNorm**, **SwiGLU**, **QK-Norm**, **gated attention** | P.10 |

## P.1. KV-cache: vì sao nó là nút cổ chai

Khi sinh token thứ $t$, mô hình cần $K, V$ của **tất cả** $t-1$ token trước. Tính lại mỗi bước rất phí → lưu lại (**KV-cache**). Kích thước:

$$\text{bytes/token} = 2 \times n_{\text{layers}} \times n_{\text{kv\_heads}} \times d_{\text{head}} \times \text{bytes/phần tử}$$

(số 2 là cho K và V). **Ví dụ số — Llama-3-8B** (32 tầng, 8 head KV, $d_{\text{head}} = 128$, bf16): $2 \times 32 \times 8 \times 128 \times 2 = 131\,072$ B = **128 KiB/token** → 8K token = 1 GiB, 128K token = **16 GiB** — lớn gấp đôi chính trọng số mô hình (~16 GB)! *(tính toán từ config; Meta, 2024)*

In [ ]:
def kv_bytes_per_token(n_layers, n_kv_heads, head_dim, bytes_per=2):
    return 2 * n_layers * n_kv_heads * head_dim * bytes_per

def fmt(b):
    for unit in ['B', 'KiB', 'MiB', 'GiB', 'TiB']:
        if b < 1024: return f"{b:,.1f} {unit}"
        b /= 1024

configs = {
    "Transformer-base (6 tầng, MHA 8×64)":         kv_bytes_per_token(6, 8, 64),
    "GPT-3 175B (96 tầng, MHA 96×128)":             kv_bytes_per_token(96, 96, 128),
    "Llama-3-8B nếu dùng MHA (32 KV head)":         kv_bytes_per_token(32, 32, 128),
    "Llama-3-8B thực tế: GQA (8 KV head)":          kv_bytes_per_token(32, 8, 128),
    "Llama-3-8B nếu dùng MQA (1 KV head)":          kv_bytes_per_token(32, 1, 128),
    # DeepSeek-V3: 61 tầng; MLA cache latent 512 + key RoPE 64 (config.json của DeepSeek-V3)
    "DeepSeek-V3 671B: MLA (512+64)/tầng":          (512 + 64) * 61 * 2,
    # MHA tương đương: 128 head, K = 128+64 chiều, V = 128 chiều
    "DeepSeek-V3 nếu dùng MHA (128 head)":          61 * 128 * (192 + 128) * 2,
}
print(f"{'Cấu hình':46s} {'/token':>12s} {'8K token':>12s} {'128K token':>12s}")
for name, b in configs.items():
    print(f"{name:46s} {fmt(b):>12s} {fmt(b * 8192):>12s} {fmt(b * 131072):>12s}")
print("\n→ Mô hình 671B dùng MLA có KV-cache/token NHỎ HƠN Llama-3-8B dùng GQA.")

## P.2. KV-cache hoạt động ra sao — kiểm chứng bằng code

Ý tưởng: với causal mask, $K, V$ của token cũ **không bao giờ thay đổi** khi thêm token mới. Vậy chỉ cần tính $q, k, v$ cho token mới, nối $k, v$ vào cache, và cho $q$ mới attend toàn bộ cache. Kết quả phải **trùng khớp** với tính lại cả chuỗi.

In [ ]:
torch.manual_seed(0)
d, T_demo = 32, 7
Wq, Wk, Wv = (nn.Linear(d, d, bias=False) for _ in range(3))
x_seq = torch.randn(1, T_demo, d)

# (a) Tính lại toàn bộ chuỗi với causal mask
full, _ = attention(Wq(x_seq), Wk(x_seq), Wv(x_seq), mask=torch.tril(torch.ones(T_demo, T_demo)))

# (b) Sinh tuần tự với KV-cache: mỗi bước chỉ tính cho 1 token mới
k_cache, v_cache, outs = [], [], []
for t in range(T_demo):
    xt = x_seq[:, t:t+1]                      # token mới
    k_cache.append(Wk(xt)); v_cache.append(Wv(xt))
    K_all, V_all = torch.cat(k_cache, 1), torch.cat(v_cache, 1)
    o, _ = attention(Wq(xt), K_all, V_all)    # không cần mask: cache chỉ chứa quá khứ
    outs.append(o)
incremental = torch.cat(outs, 1)
print("KV-cache cho kết quả trùng khớp:", torch.allclose(full, incremental, atol=1e-6))
print(f"Số phép chiếu K/V: không cache = {T_demo*(T_demo+1)//2} lần, có cache = {T_demo} lần")

## P.3. MQA và GQA — chia sẻ K/V giữa các head

- **MQA** (Shazeer, 2019 *[preprint]*): mọi head query dùng **chung một** cặp K, V → cache giảm $h$ lần, nhưng chất lượng có thể giảm.
- **GQA** (Ainslie et al., 2023): chia $H$ head query thành $G$ nhóm, mỗi nhóm chung một cặp K, V. $G = 1$ là MQA, $G = H$ là MHA. Có thể chuyển checkpoint MHA sang GQA bằng cách lấy trung bình các head K/V rồi huấn luyện tiếp chỉ với ~5% lượng tính toán gốc.

**Ví dụ số:** Llama-3-8B có 32 head query, 8 head KV → mỗi nhóm 4 head query dùng chung 1 cặp K/V → **cache giảm 4 lần** (512 → 128 KiB/token). Qwen3-235B-A22B dùng 64 head query / 4 head KV (Qwen Team, 2025 *[preprint]*).

In [ ]:
def gqa_attention(q, k, v, causal=True):
    """q: (B, Hq, T, d); k, v: (B, Hkv, T, d) với Hq chia hết cho Hkv.
    Cài đặt đơn giản: lặp mỗi head K/V cho cả nhóm query tương ứng."""
    group = q.size(1) // k.size(1)
    k = k.repeat_interleave(group, dim=1)
    v = v.repeat_interleave(group, dim=1)
    return F.scaled_dot_product_attention(q, k, v, is_causal=causal)

B, Hq, Hkv, T_, dh = 1, 32, 8, 10, 16
q, k, v = torch.randn(B, Hq, T_, dh), torch.randn(B, Hkv, T_, dh), torch.randn(B, Hkv, T_, dh)
out_gqa = gqa_attention(q, k, v)
print("Output GQA:", tuple(out_gqa.shape), "| K/V chỉ có", Hkv, "head thay vì", Hq)

# PyTorch ≥ 2.5 có tham số enable_gqa (tài liệu ghi là tính năng thử nghiệm, chủ yếu cho CUDA)
try:
    out_native = F.scaled_dot_product_attention(q, k, v, is_causal=True, enable_gqa=True)
    print("enable_gqa=True khớp cài đặt thủ công:", torch.allclose(out_gqa, out_native, atol=1e-5))
except Exception as e:
    print("enable_gqa chưa hỗ trợ ở môi trường này → dùng repeat_interleave như trên. Lý do:", type(e).__name__)

## P.4. MLA — nén K/V thay vì chia sẻ (DeepSeek)

**Multi-head Latent Attention** (DeepSeek-AI, 2024 *[preprint]*) chiếu mỗi token xuống một vector ẩn nhỏ $c_t = W^{DKV} x_t$ (DeepSeek-V3: $d_c = 512$) và **chỉ cache $c_t$** (cộng một key RoPE 64 chiều). K và V được "giải nén" từ $c_t$: $k_t = W^{UK} c_t$, $v_t = W^{UV} c_t$.

Mẹo then chốt — **hấp thụ ma trận**: $q^\top k_t = q^\top W^{UK} c_t = (W^{UK\top} q)^\top c_t$. Tức là có thể nhân trước $W^{UK}$ vào query và tính attention **trực tiếp trên $c_t$**, không bao giờ cần dựng lại K đầy đủ.

DeepSeek-V2 báo cáo giảm KV-cache 93,3% và tăng throughput 5,76× — nhưng lưu ý con số so với DeepSeek 67B (một mô hình khác), không phải MHA cùng kích thước. Phép so sánh công bằng ở P.1: MLA nhỏ hơn MHA tương đương khoảng **71 lần** *(tính toán)*.

In [ ]:
torch.manual_seed(1)
d_model_, d_c, d_head_ = 64, 8, 16
W_dkv = torch.randn(d_c, d_model_) / d_model_**0.5    # nén: d_model → d_c
W_uk = torch.randn(d_head_, d_c) / d_c**0.5            # giải nén key: d_c → d_head
x_tok = torch.randn(5, d_model_)                        # 5 token trong cache
q_new = torch.randn(d_head_)                            # query của token mới

c = x_tok @ W_dkv.T                                     # (5, d_c)  ← CHỈ LƯU CÁI NÀY
scores_naive = (c @ W_uk.T) @ q_new                     # dựng lại K rồi nhân q
scores_absorbed = c @ (W_uk.T @ q_new)                  # hấp thụ W_uk vào query
print("Hai cách tính score trùng nhau:", torch.allclose(scores_naive, scores_absorbed, atol=1e-5))
print(f"Cache mỗi token: MLA = {d_c} số | MHA 4 head K+V = {2 * 4 * d_head_} số → nhỏ hơn {2*4*d_head_//d_c}×")

## P.5. RoPE — mã hoá vị trí bằng phép quay

**Rotary Position Embedding** (Su et al., 2024) không *cộng* vector vị trí vào embedding mà **quay** từng cặp toạ độ của $q$ và $k$ một góc tỉ lệ với vị trí: cặp thứ $i$ quay góc $m\,\theta_i$ với $\theta_i = \text{base}^{-2i/d}$. Vì $R_m^\top R_n = R_{n-m}$:

$$\big(R_m q\big)^\top \big(R_n k\big) = q^\top R_{n-m}\, k \quad\Rightarrow\quad \text{score chỉ phụ thuộc khoảng cách } n - m .$$

**Ví dụ số** *(tính toán)*: $d_{\text{head}} = 128$, base = 10 000 → cặp nhanh nhất quay 1 rad/token (chu kỳ ≈ 6,3 token), cặp chậm nhất quay ≈ $1{,}15\times10^{-4}$ rad/token (chu kỳ ≈ **54 000 token**). Llama 3 tăng base lên 500 000 → chu kỳ chậm nhất ≈ **2,56 triệu token**; Qwen3 tăng lên 1 000 000 (kỹ thuật "ABF"). Tăng base = "làm đồng hồ quay chậm lại" để phân biệt được vị trí ở ngữ cảnh rất dài. Các kỹ thuật kéo dài ngữ cảnh khác: Position Interpolation (Chen et al., 2023 *[preprint]*), YaRN (Peng et al., 2024 — dùng trong Qwen, DeepSeek-V3).

In [ ]:
def rope(x, base=10000.0):
    """x: (..., T, d) với d chẵn. Quay cặp (x[2i], x[2i+1]) một góc pos · base^(-2i/d)."""
    T_, d_ = x.shape[-2:]
    theta = base ** (-torch.arange(0, d_, 2).float() / d_)          # (d/2,)
    ang = torch.arange(T_).float()[:, None] * theta[None]            # (T, d/2)
    cos, sin = ang.cos(), ang.sin()
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

torch.manual_seed(0)
d_r, T_r = 16, 30
q_vec, k_vec = torch.randn(d_r), torch.randn(d_r)
Qr = rope(q_vec.expand(T_r, d_r).clone())      # cùng một query đặt ở mọi vị trí
Kr = rope(k_vec.expand(T_r, d_r).clone())
print("Score với khoảng cách n - m = 5 tại các vị trí khác nhau:")
for m in [0, 7, 20]:
    print(f"  m = {m:2d}, n = {m+5:2d}: {float(Qr[m] @ Kr[m+5]):.5f}")
print("→ Giống hệt nhau: RoPE mã hoá VỊ TRÍ TƯƠNG ĐỐI một cách tự nhiên.")

for base in [1e4, 5e5, 1e6]:
    slowest = base ** (-(126) / 128)
    print(f"base = {base:>9,.0f}: chu kỳ quay chậm nhất (d_head=128) ≈ {2*math.pi/slowest:>12,.0f} token")

## P.6. ALiBi — không cần embedding vị trí

**ALiBi** (Press, Smith & Lewis, 2022) bỏ hẳn embedding vị trí, chỉ **cộng một phạt tuyến tính** $-m_h\,|i - j|$ vào score trước softmax, với độ dốc $m_h$ khác nhau cho từng head (8 head: $1/2, 1/4, \dots, 1/256$). Head có $m_h$ lớn nhìn gần, head có $m_h$ nhỏ nhìn xa.

**Kết quả:** mô hình 1,3B huấn luyện ở độ dài 1024 ngoại suy lên 2048 ngang mô hình sinusoidal huấn luyện thẳng ở 2048, mà nhanh hơn và tốn bộ nhớ ít hơn khoảng 11%. 🇻🇳 **PhoGPT** (Nguyen et al., 2023 *[preprint]*) — LLM tiếng Việt 3,7B của VinAI — dùng ALiBi.

In [ ]:
def alibi_bias(n_heads, T_):
    slopes = torch.tensor([2 ** (-8 * (h + 1) / n_heads) for h in range(n_heads)])
    dist = torch.arange(T_)[None, :] - torch.arange(T_)[:, None]          # j - i
    return slopes[:, None, None] * dist.clamp(max=0)                       # phạt quá khứ; tương lai đã bị causal mask che

bias = alibi_bias(8, 12)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, h in zip(axes, [0, 3, 7]):
    im = ax.imshow(bias[h], cmap='Reds_r')
    ax.set_title(f'Head {h}: slope = 1/{int(2 ** (8 * (h + 1) / 8))}')
    ax.set_xlabel('Key j'); ax.set_ylabel('Query i')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('ALiBi: phạt tuyến tính theo khoảng cách, mỗi head một độ dốc', weight='bold')
plt.tight_layout(); plt.show()

## P.7. FlashAttention — attention *chính xác*, không bao giờ ghi ma trận $T \times T$

**Vấn đề** *(tính toán)*: với $T = 8192$, 32 head, fp16, ma trận score của **một tầng** nặng $32 \times 8192^2 \times 2$ B = **4 GiB**. Đọc/ghi ma trận này ra bộ nhớ GPU (HBM) mới là thứ chậm, không phải phép nhân.

**Ý tưởng** (Dao et al., 2022): chia Q, K, V thành các khối nhỏ vừa bộ nhớ SRAM trên chip; với mỗi khối query, duyệt qua từng khối key và cập nhật softmax **"trực tuyến"** — giữ giá trị lớn nhất hiện tại $m$ và tổng $\ell$, khi gặp khối mới thì nhân lại kết quả cũ với $e^{m_{\text{cũ}} - m_{\text{mới}}}$. Bộ nhớ giảm từ $O(T^2)$ xuống $O(T)$, số FLOPs **không đổi**, kết quả **chính xác** (không xấp xỉ).

| Phiên bản | Phần cứng | Kết quả tiêu biểu |
|---|---|---|
| FlashAttention (Dao et al., 2022) | A100 | Nhanh hơn ~3× trên GPT-2 (seq 1K) |
| FlashAttention-2 (Dao, 2024) | A100 | Tới ~225 TFLOPs/s |
| FlashAttention-3 (Shah et al., 2024) | H100 | ~740 TFLOPs/s FP16; ~1,2 PFLOPs/s FP8 |
| FlashAttention-4 (Zadouri et al., 2026 *[preprint]*) | B200 | 1 613 TFLOPs/s (71% hiệu suất) |

Trong PyTorch, chỉ cần gọi `F.scaled_dot_product_attention` (Phần E) — backend FlashAttention được chọn tự động khi phần cứng hỗ trợ. Cell dưới cài **thuật toán** online softmax bằng PyTorch thuần để thấy nó cho kết quả trùng với attention thường (tốc độ thật cần kernel GPU).

In [ ]:
def tiled_attention(q, k, v, block=4):
    """Attention theo khối với online softmax (ý tưởng cốt lõi của FlashAttention). q, k, v: (T, d)."""
    T_, d_ = q.shape
    out = torch.zeros_like(q)
    for i in range(0, T_, block):
        qi = q[i:i+block] / math.sqrt(d_)
        m = torch.full((qi.size(0), 1), -float('inf'))    # max hiện tại mỗi hàng
        l = torch.zeros(qi.size(0), 1)                     # tổng e^(s - m) hiện tại
        acc = torch.zeros(qi.size(0), d_)                  # tổng có trọng số (chưa chuẩn hoá)
        for j in range(0, T_, block):
            s = qi @ k[j:j+block].T                        # chỉ một khối score nhỏ
            m_new = torch.maximum(m, s.max(-1, keepdim=True).values)
            scale = torch.exp(m - m_new)                   # hiệu chỉnh phần đã cộng trước đó
            p = torch.exp(s - m_new)
            l = l * scale + p.sum(-1, keepdim=True)
            acc = acc * scale + p @ v[j:j+block]
            m = m_new
        out[i:i+block] = acc / l
    return out

torch.manual_seed(0)
q_t, k_t, v_t = torch.randn(16, 8), torch.randn(16, 8), torch.randn(16, 8)
naive = torch.softmax(q_t @ k_t.T / math.sqrt(8), -1) @ v_t
print("Tiled (online softmax) == naive:", torch.allclose(tiled_attention(q_t, k_t, v_t), naive, atol=1e-5))
print(f"Bộ nhớ score lớn nhất cần giữ: naive = 16×16 = 256 số; tiled = 4×4 = 16 số")

## P.8. Sliding-window và sparse attention

- **Sliding-window**: mỗi token chỉ nhìn $W$ token gần nhất → chi phí $O(T\cdot W)$. Qua $L$ tầng, thông tin vẫn lan xa tới $L \times W$ token. *Mistral 7B* (Jiang et al., 2023 *[preprint]*): $W = 4096$, 32 tầng → vùng tiếp nhận lý thuyết ≈ 131K token.
- **Xen kẽ cục bộ/toàn cục**: *Gemma 2* xen kẽ 1:1 (cửa sổ 4096); *Gemma 3* dùng **5 tầng cục bộ : 1 tầng toàn cục**, cửa sổ 1024, giảm phần bộ nhớ KV-cache từ ~60% xuống dưới 15% (Gemma Team, 2025 *[preprint]*).
- **Sparse học được**: *Native Sparse Attention* của DeepSeek (Yuan et al., 2025 — giải Best Paper ACL 2025) kết hợp 3 nhánh: nén thô, chọn khối quan trọng, cửa sổ trượt. *DeepSeek Sparse Attention* trong DeepSeek-V3.2 (2025 *[preprint]*) dùng một "bộ chấm điểm" nhẹ chọn **top-k = 2048** token cho mỗi query — ở ngữ cảnh 128K, mỗi query chỉ nhìn **1,6%** số token *(tính toán)*.
- Các mẫu cố định đời đầu: Sparse Transformer (Child et al., 2019), Longformer (Beltagy et al., 2020), BigBird (Zaheer et al., 2020).

In [ ]:
T_s, W_s, topk = 16, 4, 3
causal_m = torch.tril(torch.ones(T_s, T_s, dtype=torch.bool))
window_m = causal_m & (torch.arange(T_s)[:, None] - torch.arange(T_s)[None, :] < W_s)

# Top-k động (kiểu DSA): giữ k key có điểm cao nhất theo một "bộ chấm điểm" rẻ
torch.manual_seed(3)
cheap_scores = torch.randn(T_s, T_s).masked_fill(~causal_m, -float('inf'))
idx = cheap_scores.topk(topk, dim=-1).indices
topk_m = torch.zeros(T_s, T_s, dtype=torch.bool).scatter_(1, idx, True) & causal_m

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (m_, title) in zip(axes, [(causal_m, 'Causal đầy đủ: O(T²)'),
                                  (window_m, f'Sliding window W={W_s}: O(T·W)'),
                                  (topk_m, f'Top-k động (k={topk}): O(T·k)')]):
    ax.imshow(m_, cmap='Greens'); ax.set_title(f'{title}\n{int(m_.sum())} ô được tính')
    ax.set_xlabel('Key'); ax.set_ylabel('Query')
plt.tight_layout(); plt.show()
print(f"Ở ngữ cảnh 128K với top-k = 2048: mỗi query nhìn {2048/131072:.1%} số token.")

## P.9. Linear attention và mô hình trạng thái — attention "biến thành" RNN

Nếu thay $\text{softmax}(q^\top k)$ bằng tích đặc trưng $\phi(q)^\top \phi(k)$ (Katharopoulos et al., 2020), ta đổi được thứ tự nhân: $\big(\phi(Q)\phi(K)^\top\big)V = \phi(Q)\big(\phi(K)^\top V\big)$. Với causal mask, phần trong ngoặc trở thành **trạng thái** cập nhật dần $S_t = S_{t-1} + \phi(k_t)v_t^\top$ — chính là một RNN với chi phí $O(T)$ và bộ nhớ **không tăng** theo độ dài.

Các mô hình trạng thái (SSM) như **Mamba** (Gu & Dao, 2024) và **Mamba-2** (Dao & Gu, 2024) là họ hàng gần. Xu hướng 2024–2026 là **lai (hybrid)**: phần lớn tầng rẻ, vài tầng softmax attention đầy đủ — *Jamba* (tỉ lệ 1:7), *Qwen3-Next* (3 tầng Gated DeltaNet : 1 tầng gated attention; Qwen Team, 2025 *[preprint]*).

In [ ]:
def phi(x):                      # ánh xạ đặc trưng dương: elu(x) + 1
    return F.elu(x) + 1

torch.manual_seed(0)
T_l, d_l = 12, 8
ql, kl, vl = torch.randn(T_l, d_l), torch.randn(T_l, d_l), torch.randn(T_l, d_l)

# (a) Dạng song song (dùng khi huấn luyện): ma trận T×T có mask
A = (phi(ql) @ phi(kl).T) * torch.tril(torch.ones(T_l, T_l))
parallel = (A @ vl) / A.sum(-1, keepdim=True)

# (b) Dạng hồi quy (dùng khi sinh): chỉ giữ trạng thái S (d×d) và z (d)
S, z, rec = torch.zeros(d_l, d_l), torch.zeros(d_l), []
for t in range(T_l):
    S = S + torch.outer(phi(kl[t]), vl[t]); z = z + phi(kl[t])
    rec.append((phi(ql[t]) @ S) / (phi(ql[t]) @ z))
print("Dạng song song == dạng hồi quy:", torch.allclose(parallel, torch.stack(rec), atol=1e-5))
print(f"Bộ nhớ khi sinh: softmax attention cần KV-cache tăng theo T; linear attention chỉ cần S có {d_l*d_l} số, cố định.")

## P.10. Khối Transformer năm 2025 khác bản 2017 ở mọi chi tiết

| | Transformer (2017) | Llama 3 8B (2024) | Qwen3 (2025) | DeepSeek-V3 (12/2024) |
|---|---|---|---|---|
| Chuẩn hoá | Post-LN (LayerNorm) | Pre-**RMSNorm** | Pre-RMSNorm + **QK-Norm** | Pre-RMSNorm |
| Vị trí | Sinusoidal tuyệt đối | **RoPE**, base 500K | RoPE, base 1M + YaRN | RoPE tách riêng 64 chiều, YaRN |
| Attention | MHA 8 head | **GQA** 32Q / 8KV | GQA (vd. 64Q / 4KV bản 235B) | **MLA**, 128 head |
| FFN | ReLU, $d_{ff} = 4d$ | **SwiGLU**, 4096 → 14 336 | SwiGLU; MoE | MoE 256 chuyên gia (+1 dùng chung), 8 kích hoạt |
| Bias | Có | Không | Không | Không |
| Ngữ cảnh | 512 | 8K (bản 3.1: 128K) | 32K gốc, mở rộng bằng YaRN | 128K |

*Nguồn: Vaswani et al. (2017); Meta (2024); Qwen Team (2025) [preprint]; DeepSeek-AI (2024) [preprint] và các file `config.json` công khai.*

**Ba thay đổi chính:**
1. **RMSNorm** (Zhang & Sennrich, 2019): bỏ trừ trung bình và bias → nhanh hơn 7–64% với chất lượng tương đương (đã cài ở H.1).
2. **SwiGLU** (Shazeer, 2020 *[preprint]*): $\text{FFN}(x) = W_2\big(\text{SiLU}(xW_1) \odot xW_3\big)$ — có **3** ma trận. Để giữ số tham số bằng FFN $4d$: $3 d\, d_{ff} = 8d^2 \Rightarrow d_{ff} = 8d/3$. **Ví dụ số** *(tính toán)*: $d = 4096$ → $10\,923$; Meta nhân thêm hệ số 1,3 → 14 199, làm tròn lên bội 1024 → **14 336**, đúng `intermediate_size` của Llama-3-8B.
3. **QK-Norm** (Dehghani et al., 2023): chuẩn hoá $q$ và $k$ trước tích vô hướng để score không phình to khi huấn luyện ở quy mô lớn. Gemma 3 dùng QK-Norm thay cho "soft-capping" $c\cdot\tanh(s/c)$ của Gemma 2.

**Gated attention** (Qiu et al., 2025 — giải Best Paper NeurIPS 2025): nhân đầu ra mỗi head với một cổng $\sigma(xW_\theta)$. Cổng thêm phi tuyến vào ánh xạ hạng thấp $W_O W_V$ (chính là cặp "value-down/value-up" của 3B1B), làm attention thưa hơn, giảm hiện tượng *attention sink* và cho phép learning rate lớn hơn. Được dùng trong Qwen3-Next.

Cell dưới lắp một **khối decoder kiểu 2025** thu nhỏ từ các mảnh đã học: pre-RMSNorm, RoPE, GQA, QK-Norm, gated attention, SwiGLU.

In [ ]:
class ModernDecoderBlock(nn.Module):
    """Khối decoder kiểu 2025 (thu nhỏ): pre-RMSNorm + RoPE + GQA + QK-Norm + gated attention + SwiGLU."""
    def __init__(self, d=64, n_q=8, n_kv=2, d_ff=None, gated=True):
        super().__init__()
        self.n_q, self.n_kv, self.dh = n_q, n_kv, d // n_q
        d_ff = d_ff or int(8 * d / 3)
        self.norm1, self.norm2 = RMSNorm(d), RMSNorm(d)
        self.wq = nn.Linear(d, n_q * self.dh, bias=False)
        self.wkv = nn.Linear(d, 2 * n_kv * self.dh, bias=False)
        self.q_norm, self.k_norm = RMSNorm(self.dh), RMSNorm(self.dh)          # QK-Norm
        self.gate = nn.Linear(d, n_q * self.dh, bias=False) if gated else None  # gated attention
        self.wo = nn.Linear(n_q * self.dh, d, bias=False)
        self.w1, self.w3 = nn.Linear(d, d_ff, bias=False), nn.Linear(d, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d, bias=False)

    def forward(self, x):
        B_, T_, _ = x.shape
        h = self.norm1(x)
        q = self.wq(h).view(B_, T_, self.n_q, self.dh).transpose(1, 2)
        k, v = self.wkv(h).view(B_, T_, 2, self.n_kv, self.dh).permute(2, 0, 3, 1, 4)
        q, k = rope(self.q_norm(q)), rope(self.k_norm(k))
        a = gqa_attention(q, k, v, causal=True).transpose(1, 2).reshape(B_, T_, -1)
        if self.gate is not None:
            a = a * torch.sigmoid(self.gate(h))
        x = x + self.wo(a)
        h = self.norm2(x)
        return x + self.w2(F.silu(self.w1(h)) * self.w3(h))

blk = ModernDecoderBlock()
y = blk(torch.randn(2, 10, 64))
n_modern = sum(p.numel() for p in blk.parameters())
print("Output:", tuple(y.shape), f"| tham số: {n_modern:,}")
print(f"So với 1 tầng decoder-only kiểu 2017 cùng d=64 (MHA + FFN 4d, có bias): "
      f"{4*(64*64+64) + 2*64*256 + 256 + 64 + 4*64:,} tham số")

## P.11. Bốn hiện tượng attention nên biết khi làm việc với LLM

1. **Attention sink** (Xiao et al., 2024): LLM dồn nhiều attention vào vài token đầu tiên bất kể nội dung — vì softmax buộc tổng bằng 1, head "không muốn nhìn gì" phải đổ trọng số vào đâu đó. Giữ KV của 4 token đầu + một cửa sổ gần nhất đủ để sinh ổn định tới hàng triệu token (StreamingLLM).
2. **Lost in the Middle** (Liu et al., 2024): độ chính xác truy xuất có dạng chữ **U** theo vị trí thông tin trong ngữ cảnh dài — tốt ở đầu và cuối, kém ở giữa. Hệ quả thực tế cho RAG: đặt đoạn quan trọng nhất ở đầu hoặc cuối prompt.
3. **Differential Transformer** (Ye et al., 2025): lấy hiệu hai bản đồ attention $(\text{softmax}_1 - \lambda\,\text{softmax}_2)V$ để khử "nhiễu attention".
4. **Gated attention** (Qiu et al., 2025): cổng sigmoid sau attention làm biến mất attention sink (P.10).

Cell tuỳ chọn dưới đây đo attention sink trên GPT-2 thật.

In [ ]:
# [cần Internet] Đo "attention sink" trên GPT-2 small: tỉ lệ attention đổ vào token ĐẦU TIÊN
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok_g = AutoTokenizer.from_pretrained("openai-community/gpt2")
    gpt2 = AutoModelForCausalLM.from_pretrained("openai-community/gpt2", attn_implementation="eager").eval()
    text = "The river flooded the village after three days of heavy rain, and the villagers moved to higher ground."
    with torch.no_grad():
        att = gpt2(**tok_g(text, return_tensors="pt"), output_attentions=True).attentions
    sink = [a[0, :, 5:, 0].mean().item() for a in att]      # trung bình trên head, bỏ 5 query đầu
    plt.figure(figsize=(8, 3.5)); plt.bar(range(len(sink)), sink, color='tomato')
    plt.xlabel('Tầng'); plt.ylabel('Attention vào token #0')
    plt.title('GPT-2: attention sink xuất hiện rõ từ các tầng giữa'); plt.grid(alpha=0.3, axis='y')
    plt.tight_layout(); plt.show()
except Exception as e:
    print("Bỏ qua (cần cài `transformers` và có Internet):", type(e).__name__)

---

# 🇻🇳 Q. Ứng dụng cho tiếng Việt: âm tiết, tokenizer và chi phí $O(T^2)$

> **⏱ Ý chính.**
> 1. Dấu cách trong tiếng Việt tách **âm tiết**, không tách **từ** ("Hà Nội" = 1 từ, 2 âm tiết).
> 2. Cùng một chữ có nhiều cách mã hoá Unicode và hai kiểu đặt dấu → phải **chuẩn hoá** trước khi đưa vào mô hình.
> 3. Tokenizer thiên về tiếng Anh làm câu tiếng Việt **dài hơn** $p$ lần → chi phí attention tăng **$p^2$** lần, KV-cache tăng $p$ lần.
> 4. Dấu thanh là bài toán **ngữ cảnh** thuần tuý → demo lý tưởng để huấn luyện Transformer tự cài (Q.5).

## Q.1. Âm tiết hay từ? Một tranh luận còn mở

Nhóm tác giả PhoBERT thống kê **85% loại từ tiếng Việt gồm từ hai âm tiết trở lên**; vì vậy PhoBERT tách từ bằng RDRSegmenter/VnCoreNLP trước khi áp BPE ("Hà_Nội"), và repo ghi rõ đầu vào *phải* được tách từ (Nguyen & Nguyen, 2020; Vu et al., 2018). Ngược lại, XLM-R, mBERT, ViSoBERT, ViT5 làm việc thẳng trên âm tiết.

Bằng chứng ngược chiều: Nguyen, Nguyen & Nguyen (2025, AAAI) huấn luyện từ đầu các baseline Transformer/LSTM/GRU/TextCNN trên sáu bộ dữ liệu tiếng Việt và thu được **F1 trung bình 71,85% khi *không* tách từ so với 70,49% khi tách từ**. Hai kết quả không mâu thuẫn trực tiếp (một bên là mô hình tiền huấn luyện lớn, một bên là baseline nhỏ) — đây là câu hỏi mở tốt cho đề tài sinh viên.

**Ảnh hưởng lên attention:** tách từ làm chuỗi **ngắn hơn** (ít token hơn → ma trận $T\times T$ nhỏ hơn) và mỗi token mang nghĩa trọn vẹn hơn, nhưng lỗi tách từ sẽ lan sang mọi tầng.

In [ ]:
# Q.2. Chuẩn hoá tiếng Việt: Unicode NFC + thống nhất kiểu đặt dấu + bỏ dấu
TONE_MARKS = {"́", "̀", "̉", "̃", "̣"}     # sắc, huyền, hỏi, ngã, nặng
_VALID = {("o", "a"), ("o", "e"), ("u", "y")}
_pat = re.compile(r"(?<![qQ])([oOuU])([^\W\d_])(?![^\W\d_])")

def normalize_tone_style(text):
    """Đưa về một kiểu đặt dấu: hoà→hòa, thuỷ→thủy, xoè→xòe (không đụng 'quý', 'khoảng', 'hoán')."""
    def repl(m):
        first, second = m.group(1), m.group(2)
        nfd = unicodedata.normalize("NFD", second)
        base, marks = nfd[0], nfd[1:]
        if (first.lower(), base.lower()) not in _VALID or len(marks) != 1 or marks not in TONE_MARKS:
            return m.group(0)
        return unicodedata.normalize("NFC", first + marks) + base
    return _pat.sub(repl, text)

def normalize_vi(text):
    return re.sub(r"\s+", " ", normalize_tone_style(unicodedata.normalize("NFC", text))).strip()

def remove_accents(text):
    text = "".join(c for c in unicodedata.normalize("NFD", text) if unicodedata.category(c) != "Mn")
    return text.replace("đ", "d").replace("Đ", "D")

a, b = "Thuỷ lợi", unicodedata.normalize("NFD", "Thủy lợi")
print(f"'{a}' và '{b}' trông giống nhau nhưng a == b? {a == b} | số code point: {len(a)} vs {len(b)}")
print("Sau chuẩn hoá:", normalize_vi(a) == normalize_vi(b), "→", normalize_vi(a))
for t in ["Hoà bình", "quý khách", "khoảng cách", "hoán đổi", "xoè ô"]:
    print(f"  {t:12s} → {normalize_vi(t)}")
print("Bỏ dấu:", remove_accents("Trường Đại học Thủy lợi"))

## Q.3. "Thuế token": tokenizer dài hơn $p$ lần → attention đắt hơn $p^2$ lần

Petrov et al. (2023) cho thấy cùng một nội dung dịch sang ngôn ngữ khác có thể cần số token **gấp hơn 15 lần** tiếng Anh, tuỳ tokenizer — kéo theo chi phí cao hơn, chậm hơn và ngữ cảnh hiệu dụng ngắn hơn. Tiếng Việt chịu thiệt với tokenizer tiếng Anh vì mỗi chữ có dấu ("ệ", "ữ") chiếm **2–3 byte UTF-8**, trong khi tokenizer byte-level BPE của GPT-2 hầu như không có âm tiết tiếng Việt nào trong từ vựng.

**Toán học đơn giản** *(suy luận)*: nếu chuỗi dài hơn $p$ lần thì ma trận score $T\times T$ lớn hơn $p^2$ lần, KV-cache lớn hơn $p$ lần, và ngữ cảnh cố định chứa được ít hơn $p$ lần văn bản. Với $p = 3$: attention đắt hơn 9×.

Các mô hình tiếng Việt giải quyết bằng tokenizer riêng: PhoBERT (BPE 64K trên văn bản đã tách từ), ViSoBERT (SentencePiece 15K cho mạng xã hội), PhoGPT (byte-level BPE 20 480 token học trên tiếng Việt), Vistral (mở rộng tokenizer Mistral).

In [ ]:
pairs = [("Biến đổi khí hậu làm gia tăng xâm nhập mặn ở Đồng bằng sông Cửu Long.",
          "Climate change increases saltwater intrusion in the Mekong Delta."),
         ("Hệ thống cảnh báo sạt lở bờ sông sử dụng cảm biến và học sâu.",
          "The riverbank landslide warning system uses sensors and deep learning.")]
print(f"{'':6s}{'ký tự':>8s}{'byte UTF-8':>12s}{'âm tiết/từ':>12s}")
for vi, en in pairs:
    for lang, s in [("VI", normalize_vi(vi)), ("EN", en)]:
        print(f"{lang:6s}{len(s):8d}{len(s.encode('utf-8')):12d}{len(s.split()):12d}")
print("\nTiếng Việt: số byte/ký tự cao hơn → tokenizer byte-level học trên tiếng Anh sẽ sinh nhiều token hơn.")

print("\nChi phí tương đối khi chuỗi dài hơn p lần:")
for p in [1.0, 1.5, 2.0, 3.0, 4.5]:
    print(f"  p = {p:3.1f}:  attention (T²) ×{p*p:5.1f}   KV-cache ×{p:3.1f}")

In [ ]:
# [cần Internet] Tự đo số token của cùng một cặp câu Việt–Anh với các tokenizer thật
try:
    from transformers import AutoTokenizer
    vi_s, en_s = normalize_vi(pairs[0][0]), pairs[0][1]
    for name in ["openai-community/gpt2", "FacebookAI/xlm-roberta-base", "Qwen/Qwen2.5-0.5B", "vinai/phobert-base"]:
        try:
            t = AutoTokenizer.from_pretrained(name)
            n_vi, n_en = len(t.tokenize(vi_s)), len(t.tokenize(en_s))
            note = " (PhoBERT cần tách từ trước; đây là số trên âm tiết)" if "phobert" in name else ""
            print(f"{name:32s} VI = {n_vi:3d}  EN = {n_en:3d}  p = {n_vi/n_en:.2f} → attention ×{(n_vi/n_en)**2:.1f}{note}")
        except Exception as e:
            print(f"{name:32s} lỗi tải: {type(e).__name__}")
except Exception as e:
    print("Bỏ qua (cần cài `transformers` và có Internet):", type(e).__name__)

## Q.4. Năm mô hình tiếng Việt — năm chiến lược vị trí và attention

| Mô hình | Kiến trúc / cách mã hoá vị trí | Thông số đáng nêu | Nguồn |
|---|---|---|---|
| **PhoBERT** | Encoder (RoBERTa), PE **học được**, tối đa **256** token | 20 GB văn bản (bản v2: thêm 120 GB OSCAR), BPE 64K, cần tách từ | Nguyen & Nguyen (2020) |
| **ViT5** | Encoder–decoder (T5), **relative position bias** | 310M / 866M tham số, SentencePiece 36K | Phan et al. (2022) |
| **ViDeBERTa** | DeBERTa: **disentangled attention** (tách nội dung và vị trí tương đối) | Bản base 86M tham số | Tran et al. (2023) |
| **PhoGPT** | Decoder, **ALiBi** + FlashAttention | 3,7B tham số, 32 tầng × 24 head × $d_{head}$ 128, ngữ cảnh 8192 | Nguyen et al. (2023) *[preprint]* |
| **Vistral-7B** | Mistral: **RoPE + GQA + sliding window** | Tiếp tục huấn luyện Mistral 7B cho tiếng Việt | Model card *[preprint]* |

Bảng này nối trực tiếp với các phần trước: sinusoidal (C.2) → PE học được (PhoBERT) → relative bias (ViT5) → ALiBi (PhoGPT, P.6) → RoPE + GQA (Vistral, P.3–P.5). PhoGPT với $d_{\text{model}} = 3072$, $h = 24$ là ví dụ thật cho $d_k = d_{\text{model}}/h = 128$. Giới hạn 256 token của PhoBERT cộng với chi phí $O(T^2)$ giải thích vì sao văn bản dài tiếng Việt phải **chia đoạn (chunking)**.

## Q.5. 🧪 Demo chính: khôi phục dấu tiếng Việt bằng Transformer tự cài

**Bài toán:** `"mua lu nam nay gay sat lo bo song"` → `"mùa lũ năm nay gây sạt lở bờ sông"`.

**Vì sao chọn bài toán này?**
1. **Dữ liệu tự sinh:** lấy văn bản có dấu, bỏ dấu là có cặp (đầu vào, đáp án).
2. **Căn chỉnh 1–1 theo ký tự** → cross-attention lý tưởng là **đường chéo**, rất dễ đọc heatmap.
3. **Cần ngữ cảnh:** "ma" có thể là *ma, má, mà, mả, mã, mạ* — chính là động lực của attention (D.1).
4. **Có mốc tham chiếu:** trên ~11 triệu câu, mô hình Seq2Seq đạt 97,52% độ chính xác theo từ (98,83% với beam search) và mô hình TDP dựa trên Transformer decoder đạt 98,37% (Dang & Nguyen, 2020). Mô hình của ta nhỏ và dữ liệu rất ít nên sẽ thấp hơn nhiều — mục tiêu là **hiểu cơ chế**, không phải đạt SOTA.

**Thiết kế:** mức **ký tự**; encoder đọc chuỗi không dấu, decoder sinh chuỗi có dấu (dùng lại `make_model`, `Batch`, `NoamOpt`, `LabelSmoothing` từ phần K–M). Dữ liệu: khoảng 100 câu mẫu nhúng sẵn (đủ chạy trên CPU trong vài phút); có thể thay bằng corpus lớn hơn ở Bài tập R.4.

In [ ]:
VI_CORPUS = [
    'Trường Đại học Thủy lợi đào tạo kỹ sư tài nguyên nước và môi trường.',
    'Sinh viên năm ba đang học môn xử lý ngôn ngữ tự nhiên.',
    'Hôm nay trời mưa to nên đường phố Hà Nội bị ngập nhiều nơi.',
    'Mùa lũ năm nay gây sạt lở bờ sông ở nhiều tỉnh miền Trung.',
    'Hệ thống cảnh báo sớm sử dụng cảm biến đo mực nước theo thời gian thực.',
    'Người dân được hướng dẫn di chuyển đến nơi an toàn trước khi bão đổ bộ.',
    'Mô hình học sâu có thể dự báo lượng mưa chính xác hơn phương pháp cũ.',
    'Thầy giáo giảng bài rất dễ hiểu và luôn nhiệt tình với sinh viên.',
    'Cô ấy thích đọc sách vào buổi tối sau khi làm xong việc nhà.',
    'Chúng tôi đi du lịch Đà Nẵng vào kỳ nghỉ hè năm ngoái.',
    'Ngân hàng nhà nước vừa điều chỉnh lãi suất cho vay mua nhà.',
    'Bác sĩ khuyên mọi người nên uống đủ nước và tập thể dục đều đặn.',
    'Thành phố Hồ Chí Minh là trung tâm kinh tế lớn nhất cả nước.',
    'Đồng bằng sông Cửu Long đang chịu ảnh hưởng nặng nề của xâm nhập mặn.',
    'Nông dân chuyển sang trồng lúa chịu mặn để thích ứng với biến đổi khí hậu.',
    'Nhà máy thủy điện xả lũ theo quy trình vận hành liên hồ chứa.',
    'Kỹ sư kiểm tra độ an toàn của đập trước mùa mưa bão.',
    'Học sinh tiểu học được nghỉ học vì thời tiết quá lạnh.',
    'Bà ngoại tôi nấu món canh chua cá lóc rất ngon.',
    'Anh ấy đã hoàn thành luận văn thạc sĩ về trí tuệ nhân tạo.',
    'Máy tính xử lý hàng triệu phép tính chỉ trong một giây.',
    'Trí tuệ nhân tạo đang thay đổi cách chúng ta làm việc và học tập.',
    'Giảng viên hướng dẫn sinh viên viết báo cáo khoa học đúng chuẩn.',
    'Thư viện của trường mở cửa từ bảy giờ sáng đến chín giờ tối.',
    'Tôi uống một ly cà phê sữa đá mỗi buổi sáng.',
    'Mẹ tôi đi chợ mua rau, thịt và cá cho bữa cơm gia đình.',
    'Con mèo nằm ngủ yên bình trên chiếc ghế cạnh cửa sổ.',
    'Đội tuyển bóng đá Việt Nam giành chiến thắng trong trận chung kết.',
    'Giá xăng dầu tăng nhẹ trong kỳ điều chỉnh tuần này.',
    'Công ty tuyển dụng nhiều lập trình viên có kinh nghiệm về học máy.',
    'Bài toán dịch máy yêu cầu mô hình hiểu ngữ cảnh của cả câu.',
    'Cơ chế chú ý giúp mô hình tập trung vào những từ quan trọng.',
    'Mỗi đầu chú ý học một kiểu quan hệ khác nhau giữa các từ.',
    'Dữ liệu huấn luyện càng nhiều thì mô hình càng tổng quát tốt hơn.',
    'Chúng ta cần chuẩn hóa văn bản trước khi đưa vào mô hình.',
    'Tiếng Việt có sáu thanh điệu và nhiều nguyên âm có dấu.',
    'Người nông dân thu hoạch lúa sớm để tránh đợt mưa lớn sắp tới.',
    'Chính quyền địa phương huy động lực lượng gia cố đê điều.',
    'Nước sông dâng cao làm ngập hàng trăm hecta hoa màu.',
    'Các nhà khoa học đang nghiên cứu giống lúa mới năng suất cao.',
    'Buổi hội thảo thu hút nhiều chuyên gia trong và ngoài nước.',
    'Em gái tôi đang ôn thi đại học ở nhà.',
    'Chiếc xe buýt đến muộn mười lăm phút vì tắc đường.',
    'Quán phở đầu ngõ lúc nào cũng đông khách vào buổi sáng.',
    'Bệnh viện tuyến tỉnh được đầu tư thêm nhiều trang thiết bị hiện đại.',
    'Sinh viên tham gia tình nguyện dọn rác ven biển vào cuối tuần.',
    'Cô giáo khen cả lớp vì đã làm bài kiểm tra rất tốt.',
    'Mô hình ngôn ngữ lớn có thể trả lời câu hỏi bằng tiếng Việt.',
    'Nhóm nghiên cứu công bố bài báo tại hội nghị quốc tế.',
    'Hồ chứa nước cung cấp nước tưới cho hàng nghìn héc ta lúa.',
    'Trạm quan trắc ghi nhận lượng mưa vượt mức báo động ba.',
    'Ông tôi thường kể chuyện ngày xưa cho các cháu nghe.',
    'Những cánh đồng lúa chín vàng trải dài đến tận chân trời.',
    'Học máy là một nhánh quan trọng của trí tuệ nhân tạo.',
    'Mạng nơ ron sâu gồm nhiều tầng biến đổi phi tuyến.',
    'Tôi muốn học thêm tiếng Anh để đọc tài liệu chuyên ngành.',
    'Các bạn nhớ nộp bài tập trước thứ sáu tuần sau.',
    'Thời tiết miền Bắc chuyển lạnh do ảnh hưởng của không khí lạnh.',
    'Ngành du lịch phục hồi mạnh mẽ sau đại dịch.',
    'Cầu mới giúp người dân hai bên sông đi lại thuận tiện hơn.',
    'Anh trai tôi làm kỹ sư xây dựng tại một công ty lớn.',
    'Chúng tôi trồng cây xanh quanh sân trường để tạo bóng mát.',
    'Nhiều gia đình đã chuẩn bị lương thực trước mùa mưa bão.',
    'Công nghệ cảm biến giúp giám sát chất lượng nước tự động.',
    'Mô hình dự báo được huấn luyện trên dữ liệu khí tượng nhiều năm.',
    'Sáng nay tôi dậy sớm để chạy bộ quanh hồ.',
    'Bạn có biết đường đến nhà ga không.',
    'Tôi rất vui khi được gặp lại các bạn cũ.',
    'Chị ấy mua một chiếc áo mới để đi dự tiệc cưới.',
    'Buổi chiều trời nắng nhẹ, gió thổi mát rượi.',
    'Giáo sư trình bày kết quả nghiên cứu trước hội đồng khoa học.',
    'Người dân vùng lũ cần được hỗ trợ lương thực và nước sạch.',
    'Điện thoại thông minh giúp việc học trực tuyến trở nên dễ dàng hơn.',
    'Máy bơm được lắp đặt để tiêu thoát nước cho khu dân cư.',
    'Tỉnh đã ban hành kế hoạch phòng chống thiên tai năm nay.',
    'Nhà trường tổ chức lễ tốt nghiệp cho sinh viên khóa mới.',
    'Việc tách từ tiếng Việt ảnh hưởng đến chất lượng mô hình.',
    'Dấu thanh giúp phân biệt nghĩa của các từ có cùng phụ âm và nguyên âm.',
    'Chữ ma có thể thành má, mà, mả, mã hoặc mạ tùy ngữ cảnh.',
    'Mỗi buổi tối cả nhà quây quần bên mâm cơm.',
    'Tôi đã đọc xong cuốn sách về lịch sử Việt Nam.',
    'Học sinh được hướng dẫn cách phòng tránh đuối nước.',
    'Hệ thống thủy lợi giúp điều tiết nước cho sản xuất nông nghiệp.',
    'Đê biển được nâng cấp để chống lại triều cường.',
    'Các em nhỏ vui chơi trong công viên gần nhà.',
    'Nhà khoa học trẻ nhận giải thưởng về công nghệ môi trường.',
    'Lượng nước trong hồ giảm mạnh do nắng nóng kéo dài.',
    'Cơ quan khí tượng dự báo có mưa rào và dông vài nơi.',
    'Hà Nội có nhiều di tích lịch sử và văn hóa nổi tiếng.',
    'Tôi thường nghe nhạc khi làm việc để tập trung hơn.',
    'Ban tổ chức đã chuẩn bị chu đáo cho sự kiện.',
    'Bác nông dân dắt trâu ra đồng từ sáng sớm.',
    'Chúng ta phải bảo vệ nguồn nước sạch cho thế hệ sau.',
    'Phòng thí nghiệm được trang bị máy tính có card đồ họa mạnh.',
    'Mô hình Transformer xử lý toàn bộ câu cùng một lúc.',
    'Sinh viên cần hiểu rõ công thức trước khi viết mã.',
    'Kết quả thực nghiệm cho thấy phương pháp mới tốt hơn.',
    'Người lao động được nghỉ ba ngày vào dịp lễ quốc khánh.',
    'Chiếc thuyền nhỏ trôi chậm trên dòng sông yên ả.',
    'Tôi sẽ gọi điện cho bạn khi về đến nhà.',
    'Hàng cây ven đường bị gió bão quật đổ.',
    'Đoàn công tác đã khảo sát tình hình sạt lở tại địa phương.',
    'Ngôi trường mới được xây dựng khang trang và hiện đại.',
]

sents = [normalize_vi(s).lower() for s in VI_CORPUS]
rng = random.Random(0)
rng.shuffle(sents)
test_sents, train_sents = sents[:12], sents[12:]

PAD, BOS = 0, 1
tgt_itos = ['<pad>', '<s>'] + sorted(set(''.join(sents)))
src_itos = ['<pad>', '<s>'] + sorted(set(remove_accents(''.join(sents))))
tgt_stoi = {c: i for i, c in enumerate(tgt_itos)}
src_stoi = {c: i for i, c in enumerate(src_itos)}
print(f"{len(train_sents)} câu train, {len(test_sents)} câu test | vocab nguồn = {len(src_itos)}, vocab đích = {len(tgt_itos)}")
print("Ví dụ cặp dữ liệu:")
print("  vào:", remove_accents(train_sents[0]))
print("  ra :", train_sents[0])

# Mỗi ký tự không dấu được phép sinh những ký tự có dấu nào? (dùng cho giải mã có ràng buộc)
allowed = torch.zeros(len(src_itos), len(tgt_itos), dtype=torch.bool)
for c, i in tgt_stoi.items():
    if i > 1:
        allowed[src_stoi[remove_accents(c)], i] = True
print(f"\nVí dụ: 'a' có thể thành {[tgt_itos[j] for j in allowed[src_stoi['a']].nonzero().flatten().tolist()]}")

In [ ]:
L_WIN = 32   # độ dài cửa sổ ký tự

def encode_pair(text):
    src = [src_stoi[c] for c in remove_accents(text)]
    tgt = [BOS] + [tgt_stoi[c] for c in text]
    return src, tgt

def diacritic_batches(corpus, batch_size, n_batches, rng):
    """Sinh batch từ các cửa sổ ngẫu nhiên dài ≤ L_WIN ký tự của các câu."""
    for _ in range(n_batches):
        srcs, tgts = [], []
        for _ in range(batch_size):
            s = rng.choice(corpus)
            start = rng.randint(0, max(0, len(s) - L_WIN))
            src, tgt = encode_pair(s[start:start + L_WIN])
            srcs.append(src); tgts.append(tgt)
        S = torch.zeros(batch_size, L_WIN, dtype=torch.long)
        Tt = torch.zeros(batch_size, L_WIN + 1, dtype=torch.long)
        for i, (s_, t_) in enumerate(zip(srcs, tgts)):
            S[i, :len(s_)] = torch.tensor(s_); Tt[i, :len(t_)] = torch.tensor(t_)
        yield Batch(S, Tt, pad=PAD)

torch.manual_seed(0)
dia_model = make_model(len(src_itos), len(tgt_itos), N=2, d_model=64, d_ff=256, h=4, dropout=0.2)
dia_crit = LabelSmoothing(size=len(tgt_itos), padding_idx=PAD, smoothing=0.1)
dia_opt = NoamOpt(64, factor=0.5, warmup=300,
                  optimizer=torch.optim.Adam(dia_model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9))
print(f"Tham số mô hình khôi phục dấu: {sum(p.numel() for p in dia_model.parameters()):,}")

data_rng = random.Random(1)
dia_losses = []
t0 = time.time()
for epoch in range(25):
    dia_model.train()
    loss = run_epoch(diacritic_batches(train_sents, 64, 30, data_rng), dia_model,
                     SimpleLossCompute(dia_model.generator, dia_crit, dia_opt), verbose=False)
    dia_losses.append(float(loss))
    if epoch % 6 == 0 or epoch == 24:
        print(f"epoch {epoch:2d} | loss/token = {float(loss):.3f} | {time.time() - t0:5.1f}s")

In [ ]:
def restore_chunk(model, text, constrained=True):
    """Giải mã tham lam; nếu constrained, chỉ cho phép ký tự có cùng dạng không dấu với ký tự vào."""
    src_ids = [src_stoi[c] for c in text]
    src = torch.tensor([src_ids]); src_mask = torch.ones(1, 1, len(src_ids))
    memory = model.encode(src, src_mask)
    ys = torch.tensor([[BOS]])
    for i in range(len(src_ids)):
        out = model.decode(memory, src_mask, ys, subsequent_mask(ys.size(1)).type_as(src))
        logp = model.generator(out[:, -1])[0]
        if constrained:
            logp = logp.masked_fill(~allowed[src_ids[i]], -float('inf'))
        ys = torch.cat([ys, logp.argmax().view(1, 1)], dim=1)
    return ''.join(tgt_itos[j] for j in ys[0, 1:].tolist())

def restore(model, sentence, constrained=True):
    """Chia câu thành các đoạn ≤ L_WIN ký tự theo ranh giới âm tiết rồi khôi phục từng đoạn."""
    chunks, cur = [], ''
    for w in remove_accents(sentence).split():
        if cur and len(cur) + 1 + len(w) > L_WIN:
            chunks.append(cur); cur = w
        else:
            cur = f"{cur} {w}" if cur else w
    chunks.append(cur)
    return ' '.join(restore_chunk(model, c, constrained) for c in chunks)

# Baseline không có ngữ cảnh: mỗi âm tiết không dấu → dạng có dấu gặp nhiều nhất trong tập train
freq = defaultdict(Counter)
for s in train_sents:
    for w in s.split():
        freq[remove_accents(w)][w] += 1
def restore_baseline(sentence):
    return ' '.join(freq[w].most_common(1)[0][0] if w in freq else w for w in remove_accents(sentence).split())

def syllable_acc(pred, gold):
    p, g = pred.split(), gold.split()
    return sum(a == b for a, b in zip(p, g)) / len(g)

dia_model.eval()
scores = {'Baseline tần suất (không ngữ cảnh)': [], 'Transformer (giải mã tự do)': [], 'Transformer + ràng buộc': []}
with torch.no_grad():
    for s in test_sents:
        scores['Baseline tần suất (không ngữ cảnh)'].append(syllable_acc(restore_baseline(s), s))
        scores['Transformer (giải mã tự do)'].append(syllable_acc(restore(dia_model, s, constrained=False), s))
        scores['Transformer + ràng buộc'].append(syllable_acc(restore(dia_model, s, constrained=True), s))
print("Độ chính xác theo âm tiết trên câu TEST (chưa thấy khi huấn luyện):")
for k_, v_ in scores.items():
    print(f"  {k_:38s}: {np.mean(v_):.1%}")

print("\nVí dụ:")
with torch.no_grad():
    for s in test_sents[:4]:
        print("  vào     :", remove_accents(s))
        print("  mô hình :", restore(dia_model, s))
        print("  đáp án  :", s, "\n")

**Đọc kết quả (một lần chạy mẫu trên CPU: baseline ≈ 61%, Transformer tự do ≈ 45%, Transformer + ràng buộc ≈ 54%).** Với chưa đến 100 câu huấn luyện, **baseline tần suất thắng** — và đó là một bài học quan trọng, không phải lỗi:
- **Luôn so với một baseline đơn giản.** Tra từ điển âm tiết đã gặp là rất mạnh khi dữ liệu ít; mô hình mức ký tự phải *tự học* cả cách viết âm tiết tiếng Việt từ ~5 000 ký tự văn bản.
- **Attention cần dữ liệu.** Các hệ thống công bố huấn luyện trên khoảng 11 triệu câu đạt 97–99% độ chính xác theo từ (Dang & Nguyen, 2020); baseline từ điển thì không bao giờ phân biệt được *ma/má/mà/mả* trong cùng một ngữ cảnh mới. Bài tập R.4 yêu cầu tăng dữ liệu để tìm "điểm giao cắt".
- **Giải mã có ràng buộc** (chỉ cho phép "a" thành a/á/à/ả/ã/ạ/ă/â/…) cải thiện rõ rệt: đưa **tri thức miền** vào lúc giải mã mà không cần huấn luyện lại.

Tiếp theo, xem cross-attention: với bài toán căn chỉnh 1–1, ta kỳ vọng **đường chéo**.

In [ ]:
sample = remove_accents(test_sents[0])[:L_WIN].rsplit(' ', 1)[0]
src = torch.tensor([[src_stoi[c] for c in sample]])
src_mask = torch.ones(1, 1, src.size(1))
with torch.no_grad():
    pred = restore_chunk(dia_model, sample)
    tgt_in = torch.tensor([[BOS] + [tgt_stoi[c] for c in pred[:-1]]])
    _ = dia_model(src, tgt_in, src_mask, subsequent_mask(tgt_in.size(1)).type_as(src))
cross = dia_model.decoder.layers[-1].src_attn.attn[0]          # (h, T_tgt, T_src)

fig, axes = plt.subplots(1, 4, figsize=(22, 5.5))
for h in range(4):
    axes[h].imshow(cross[h].numpy(), cmap='Blues')
    axes[h].set_xticks(range(len(sample))); axes[h].set_xticklabels(list(sample), fontsize=8)
    axes[h].set_yticks(range(len(pred))); axes[h].set_yticklabels(list(pred), fontsize=8)
    axes[h].set_title(f'Cross-attention head {h + 1}'); axes[h].set_xlabel('Nguồn (không dấu)')
axes[0].set_ylabel('Đích đang sinh (có dấu)')
plt.suptitle(f'Khôi phục dấu: "{sample}" → "{pred}"', weight='bold')
plt.tight_layout(); plt.show()
print("Head có đường chéo sáng = 'nhìn đúng ký tự nguồn tương ứng'; head nhìn lệch ±vài ký tự = dùng ngữ cảnh lân cận để chọn dấu.")

## Q.6. Demo mở rộng: dịch máy Anh → Việt

Dịch máy là bài toán gốc của Transformer. Cross-attention ở đây **không** đơn điệu như khôi phục dấu (tính từ tiếng Việt đứng *sau* danh từ: "*blue* house" → "ngôi nhà *xanh*"), nên heatmap thú vị hơn nhưng khó đọc hơn.

| Hệ thống | Dữ liệu | BLEU En→Vi | Nguồn |
|---|---|---|---|
| LSTM + attention (đơn) | IWSLT'15 (133K cặp), tst2013 | 23,3 | Luong & Manning (2015) |
| LSTM + attention (ensemble 9) | IWSLT'15, tst2013 | 26,9 | Luong & Manning (2015) |
| Transformer-base | IWSLT'15 | ≈ 28,5 | repo cộng đồng *[preprint]* |
| Transformer-base | **PhoMT** (≈ 3 triệu cặp) | 42,12 | Doan et al. (2021) |
| mBART | PhoMT | 43,46 | Doan et al. (2021) |

Hai bài học: (1) Transformer vượt LSTM + attention trên cùng dữ liệu; (2) **dữ liệu** tăng ~22 lần (IWSLT → PhoMT) cho bước nhảy BLEU lớn hơn nhiều so với đổi kiến trúc. Bộ IWSLT'15 có trên Hugging Face (`IWSLT/mt_eng_vietnamese`, 133 318 / 1 269 / 1 269 cặp); repo này dùng *loading script* nên có thể không tải được với thư viện `datasets` bản mới. Mô hình nhỏ huấn luyện trên CPU sẽ cho BLEU thấp hơn nhiều — điều đó bình thường.

In [ ]:
# [cần Internet] Xem nhanh dữ liệu IWSLT'15 En–Vi và phân bố độ dài (dùng cho Bài tập R.5)
try:
    from datasets import load_dataset
    ds = load_dataset("IWSLT/mt_eng_vietnamese", "iwslt2015-en-vi", split="train[:2000]")
    ex = ds[0]["translation"]
    print("EN:", ex["en"]); print("VI:", ex["vi"])
    len_en = [len(d["translation"]["en"].split()) for d in ds]
    len_vi = [len(d["translation"]["vi"].split()) for d in ds]
    print(f"Độ dài trung bình: EN = {np.mean(len_en):.1f} từ, VI = {np.mean(len_vi):.1f} âm tiết")
except Exception as e:
    print("Không tải được IWSLT'15 (cần Internet; repo dùng loading script):", type(e).__name__)

## Q.7. Đề tài mở: attention của mô hình tiếng Việt học được gì?

Chúng tôi **không tìm thấy** nghiên cứu bình duyệt nào phân tích các attention head của PhoBERT hay mô hình tiếng Việt theo cách Clark et al. (2019) làm với BERT tiếng Anh. Đây là một đề tài nhỏ có thể phát triển thành bài báo:

- Bật `output_attentions=True` cho `vinai/phobert-base` (câu đã tách từ) và `FacebookAI/xlm-roberta-base` (câu âm tiết).
- Đo "attention nội từ": tỉ lệ attention giữa các subword/âm tiết của **cùng một từ ghép** (ví dụ "Đồng", "bằng" trong "Đồng bằng").
- So sánh theo tầng/head; kiểm tra xem XLM-R có "tự học" ranh giới từ không.
- Khung phương pháp tham khảo: nghiên cứu tương tự cho tiếng Trung (Wang et al., 2020, COLING).

In [ ]:
# [cần Internet] Khởi đầu cho đề tài Q.7: attention nội từ trên XLM-R
try:
    from transformers import AutoTokenizer, AutoModel
    tk = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")
    mdl = AutoModel.from_pretrained("FacebookAI/xlm-roberta-base", attn_implementation="eager").eval()
    words = ["Đồng bằng", "sông", "Cửu Long", "bị", "xâm nhập", "mặn"]           # từ (có thể nhiều âm tiết)
    syll = [s for w in words for s in w.split()]
    word_of = [wi for wi, w in enumerate(words) for _ in w.split()]
    enc = tk(syll, is_split_into_words=True, return_tensors="pt")
    wid = enc.word_ids()                                                           # chỉ số âm tiết của từng token
    with torch.no_grad():
        att = mdl(**enc, output_attentions=True).attentions
    same_word = torch.tensor([[wid[i] is not None and wid[j] is not None and wid[i] != wid[j]
                               and word_of[wid[i]] == word_of[wid[j]] for j in range(len(wid))] for i in range(len(wid))])
    ratio = [float(a[0].mean(0)[same_word].mean()) if same_word.any() else 0 for a in att]
    plt.figure(figsize=(8, 3.5)); plt.plot(ratio, marker='o')
    plt.xlabel('Tầng'); plt.ylabel('Attention TB giữa 2 âm tiết cùng từ')
    plt.title('XLM-R: âm tiết trong cùng từ ghép có "nhìn" nhau nhiều hơn?'); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
except Exception as e:
    print("Bỏ qua (cần cài `transformers` và có Internet):", type(e).__name__)

---

# 📝 R. Bài tập và câu hỏi thảo luận

**R.1 (Cơ bản) — Tính tay.** Làm lại ví dụ D.4b cho từ "ngọt" (hàng 1). Sau đó đổi $\vec K_{\text{ngọt}} = [4, 2]$: trọng số attention của "đường" vào "ngọt" thay đổi thế nào? Giải thích bằng ý nghĩa của $\sqrt{d_k}$.

**R.2 (Cơ bản) — Đếm tham số và bộ nhớ.** (a) Dùng công thức $12Ld^2$ ước lượng tham số của một mô hình 32 tầng, $d = 4096$. (b) Tính KV-cache/token và cho ngữ cảnh 32K của mô hình đó nếu dùng MHA 32 head, GQA 8 head, MQA. (c) Kiểm tra bằng hàm `kv_bytes_per_token`.

**R.3 (Trung bình) — Thay đổi vị trí.** Trong `make_model`, thay `PositionalEncoding` bằng (a) PE học được (`nn.Embedding(max_len, d_model)`), (b) không có PE. Huấn luyện lại bài toán copy (N) và so sánh loss. Vì sao bài toán copy vẫn học được khi không có PE ở encoder? (Gợi ý: causal mask của decoder mang thông tin vị trí — Kazemnejad et al., 2023.)

**R.4 (Trung bình) — Khôi phục dấu với nhiều dữ liệu hơn.** Thay `VI_CORPUS` bằng ≥ 20 000 câu tiếng Việt (ví dụ phía tiếng Việt của IWSLT'15 hoặc văn bản báo chí tự thu thập, nhớ `normalize_vi`). Báo cáo độ chính xác theo âm tiết của baseline tần suất và Transformer; vẽ đường học. Ở bao nhiêu câu thì Transformer vượt baseline?

**R.5 (Nâng cao) — Dịch Anh → Việt.** Dùng 20 000 cặp IWSLT'15, tokenizer mức âm tiết/từ đơn giản, huấn luyện `make_model(N=3, d_model=256)`. Vẽ cross-attention cho câu "the blue house is near the river" và chỉ ra chỗ trật tự tính từ–danh từ bị đảo.

**R.6 (Nâng cao) — Khối hiện đại.** Thay `EncoderLayer`/`DecoderLayer` bằng pre-RMSNorm + SwiGLU + RoPE (tái sử dụng P.5, P.10) và so sánh tốc độ hội tụ trên bài toán khôi phục dấu. Thử bỏ warmup với post-LN và pre-LN để kiểm chứng Xiong et al. (2020).

**R.7 (Đề tài mở) — Attention nội từ.** Thực hiện đề tài Q.7 cho PhoBERT và XLM-R, viết báo cáo 4 trang.

**Câu hỏi thảo luận**
1. Vì sao attention *không* phải là lời giải thích đáng tin cậy cho dự đoán của mô hình? (Jain & Wallace, 2019; Wiegreffe & Pinter, 2019)
2. GQA và MLA đều giảm KV-cache. Cái giá của mỗi cách là gì?
3. Với chatbot tiếng Việt dùng API tính phí theo token, "thuế token" ảnh hưởng thế nào đến chi phí và độ dài ngữ cảnh hiệu dụng?
4. Khi nào nên tách từ tiếng Việt trước khi đưa vào Transformer? Hãy lập luận từ cả chi phí attention lẫn độ chính xác.

---

# 🏁 O. Tổng kết & Hướng đi tiếp theo

## O.1. Những gì chúng ta đã đi qua

| Phần | Thành phần | Vai trò |
|------|-----------|---------|
| C | `Embeddings` + `PositionalEncoding` | Biểu diễn token + thứ tự |
| D | Trực giác Q / K / V + ví dụ tính tay | "Câu hỏi – Thẻ – Thông tin" |
| E | `attention()` (scaled dot-product) + SDPA | Công thức cốt lõi |
| F | `MultiHeadedAttention` | $h$ head song song |
| G | `PositionwiseFeedForward` | Phi tuyến độc lập theo vị trí |
| H | `LayerNorm` / `RMSNorm` + `SublayerConnection` | Ổn định huấn luyện |
| I–K | `Encoder`, `Decoder`, `EncoderDecoder`, `make_model` | Lắp ráp đầy đủ |
| L | Đếm tham số Transformer-base, GPT-3 | Hiểu quy mô |
| M–N | `Batch`, `NoamOpt`, `LabelSmoothing`, copy task | Huấn luyện end-to-end |
| P | KV-cache, GQA, MLA, RoPE, ALiBi, FlashAttention, sparse, linear, khối 2025 | Attention hiện đại |
| Q | Chuẩn hoá, thuế token, mô hình Việt, khôi phục dấu | Tiếng Việt |

## O.2. Công thức tổng kết

$$
\boxed{\;\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V\;}
\qquad
\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)\, W^O
$$

Một tầng encoder, hai cách đặt chuẩn hoá (đính chính so với bản trước — code trong notebook là **pre-LN**):

$$
\text{Post-LN (bài báo gốc):}\quad x' = \text{LN}\big(x + \text{MHA}(x)\big),\quad y = \text{LN}\big(x' + \text{FFN}(x')\big)
$$
$$
\text{Pre-LN (code này, LLM hiện đại):}\quad x' = x + \text{MHA}(\text{LN}(x)),\quad y = x' + \text{FFN}(\text{LN}(x'))
$$

## O.3. Ba bài học lớn

1. **Tách "ai chú ý ai" (QK) khỏi "thông tin gì được truyền" (OV).**
2. **Song song hoá là vua khi huấn luyện; bộ nhớ là vua khi suy luận.** 2017 tối ưu cái thứ nhất; 2023–2026 tối ưu cái thứ hai (KV-cache, FlashAttention, GQA/MLA, sparse).
3. **Với tiếng Việt, chọn tokenizer và quyết định tách từ là quyết định về *chi phí attention*,** không chỉ về độ chính xác.

## O.4. Hướng đi tiếp

| Hướng | Tóm tắt | Tài liệu gợi ý |
|-------|---------|---------|
| Kiến trúc biến thể | BERT (encoder), GPT (decoder), T5/BART (encoder–decoder), ViT cho ảnh | Devlin et al. (2019); Raffel et al. (2020) |
| Hiệu quả tính toán | FlashAttention, sparse học được (NSA, DSA), linear/SSM, hybrid | P.7–P.9 |
| Ngữ cảnh dài | RoPE + YaRN, ALiBi, attention sink | P.5, P.6, P.11 |
| Interpretability | QK/OV circuit, induction head | Elhage et al. (2021); Olsson et al. (2022) |
| Fine-tuning hiệu quả | LoRA, QLoRA — chèn ma trận hạng thấp vào $W_Q, W_V$ | Hu et al. (2022) |
| Tự xây GPT nhỏ | *Let's build GPT* và nanoGPT của Karpathy (repo đã được tác giả đánh dấu *deprecated* từ 11/2025, khuyến nghị chuyển sang **nanochat**) | Karpathy |

> Cảm ơn các bạn đã theo dõi! Hãy mạnh dạn thử nghiệm — đặc biệt với các đề tài tiếng Việt ở R.4–R.7. 🚀

---

# 📚 Tài liệu tham khảo

*Ký hiệu [preprint]: chưa qua bình duyệt (arXiv, báo cáo kỹ thuật, model card, blog, repo). Số liệu trong notebook lấy từ các nguồn này; các con số ghi "tính toán" là phép tính từ cấu hình công bố.*

### Nền tảng attention và Transformer
- Bahdanau, D., Cho, K., & Bengio, Y. (2015). Neural machine translation by jointly learning to align and translate. *ICLR 2015*. arXiv:1409.0473.
- Luong, M.-T., Pham, H., & Manning, C. D. (2015). Effective approaches to attention-based neural machine translation. *EMNLP 2015*, 1412–1421. https://aclanthology.org/D15-1166/
- Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł., & Polosukhin, I. (2017). Attention is all you need. *Advances in Neural Information Processing Systems 30 (NIPS 2017)*. arXiv:1706.03762.
- Rush, A. M. (2018). The Annotated Transformer. Harvard NLP. https://nlp.seas.harvard.edu/2018/04/03/attention.html *[preprint]*
- Huang, A., Subramanian, S., Sum, J., Almubarak, K., & Biderman, S. (2022). The Annotated Transformer (v2022). https://nlp.seas.harvard.edu/annotated-transformer/ *[preprint]*
- Sanderson, G. (3Blue1Brown). (2024, April 7). Attention in transformers, step-by-step — Deep Learning Chapter 6. https://www.3blue1brown.com/lessons/attention *[preprint]*
- Brown, T. B., et al. (2020). Language models are few-shot learners. *NeurIPS 2020*. arXiv:2005.14165.
- Press, O., & Wolf, L. (2017). Using the output embedding to improve language models. *EACL 2017*. arXiv:1608.05859.

### Chuẩn hoá, tối ưu, huấn luyện
- Ba, J. L., Kiros, J. R., & Hinton, G. E. (2016). Layer normalization. arXiv:1607.06450. *[preprint]*
- He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. *CVPR 2016*, 770–778.
- Xiong, R., Yang, Y., He, D., Zheng, K., Zheng, S., Xing, C., Zhang, H., Lan, Y., Wang, L., & Liu, T.-Y. (2020). On layer normalization in the Transformer architecture. *ICML 2020*. arXiv:2002.04745.
- Zhang, B., & Sennrich, R. (2019). Root mean square layer normalization. *NeurIPS 2019*. arXiv:1910.07467.
- Kingma, D. P., & Ba, J. (2015). Adam: A method for stochastic optimization. *ICLR 2015*. arXiv:1412.6980.
- Szegedy, C., Vanhoucke, V., Ioffe, S., Shlens, J., & Wojna, Z. (2016). Rethinking the Inception architecture for computer vision. *CVPR 2016*, 2818–2826.
- Müller, R., Kornblith, S., & Hinton, G. (2019). When does label smoothing help? *NeurIPS 2019*. arXiv:1906.02629.
- Glorot, X., & Bengio, Y. (2010). Understanding the difficulty of training deep feedforward neural networks. *AISTATS 2010*, PMLR 9:249–256.
- Williams, R. J., & Zipser, D. (1989). A learning algorithm for continually running fully recurrent neural networks. *Neural Computation*, 1(2), 270–280.

### Diễn giải attention
- Clark, K., Khandelwal, U., Levy, O., & Manning, C. D. (2019). What does BERT look at? An analysis of BERT's attention. *BlackboxNLP@ACL 2019*, 276–286.
- Voita, E., Talbot, D., Moiseev, F., Sennrich, R., & Titov, I. (2019). Analyzing multi-head self-attention: Specialized heads do the heavy lifting, the rest can be pruned. *ACL 2019*, 5797–5808.
- Michel, P., Levy, O., & Neubig, G. (2019). Are sixteen heads really better than one? *NeurIPS 2019*. arXiv:1905.10650.
- Jain, S., & Wallace, B. C. (2019). Attention is not explanation. *NAACL-HLT 2019*, 3543–3556.
- Wiegreffe, S., & Pinter, Y. (2019). Attention is not not explanation. *EMNLP-IJCNLP 2019*, 11–20.
- Geva, M., Schuster, R., Berant, J., & Levy, O. (2021). Transformer feed-forward layers are key-value memories. *EMNLP 2021*, 5484–5495.
- Elhage, N., Nanda, N., Olsson, C., Henighan, T., et al. (2021). A mathematical framework for Transformer circuits. *Transformer Circuits Thread*. https://transformer-circuits.pub/2021/framework/index.html *[preprint]*
- Olsson, C., Elhage, N., Nanda, N., Joseph, N., et al. (2022). In-context learning and induction heads. *Transformer Circuits Thread*. *[preprint]*

### Attention hiện đại (2019–2026)
- Su, J., Lu, Y., Pan, S., Murtadha, A., Wen, B., & Liu, Y. (2024). RoFormer: Enhanced Transformer with rotary position embedding. *Neurocomputing*, 568, 127063. arXiv:2104.09864.
- Press, O., Smith, N. A., & Lewis, M. (2022). Train short, test long: Attention with linear biases enables input length extrapolation. *ICLR 2022*. arXiv:2108.12409.
- Chen, S., Wong, S., Chen, L., & Tian, Y. (2023). Extending context window of large language models via positional interpolation. arXiv:2306.15595. *[preprint]*
- Peng, B., Quesnelle, J., Fan, H., & Shippole, E. (2024). YaRN: Efficient context window extension of large language models. *ICLR 2024*. arXiv:2309.00071.
- Kazemnejad, A., Padhi, I., Natesan Ramamurthy, K., Das, P., & Reddy, S. (2023). The impact of positional encoding on length generalization in Transformers. *NeurIPS 2023*. arXiv:2305.19466.
- Shazeer, N. (2019). Fast Transformer decoding: One write-head is all you need. arXiv:1911.02150. *[preprint]*
- Ainslie, J., Lee-Thorp, J., de Jong, M., Zemlyanskiy, Y., Lebrón, F., & Sanghai, S. (2023). GQA: Training generalized multi-query Transformer models from multi-head checkpoints. *EMNLP 2023*. arXiv:2305.13245.
- DeepSeek-AI. (2024). DeepSeek-V2: A strong, economical, and efficient mixture-of-experts language model. arXiv:2405.04434. *[preprint]*
- DeepSeek-AI. (2024). DeepSeek-V3 technical report. arXiv:2412.19437. *[preprint]*
- DeepSeek-AI. (2025). DeepSeek-V3.2 (DeepSeek Sparse Attention). arXiv:2512.02556. *[preprint]*
- Dao, T., Fu, D. Y., Ermon, S., Rudra, A., & Ré, C. (2022). FlashAttention: Fast and memory-efficient exact attention with IO-awareness. *NeurIPS 2022*. arXiv:2205.14135.
- Dao, T. (2024). FlashAttention-2: Faster attention with better parallelism and work partitioning. *ICLR 2024*. arXiv:2307.08691.
- Shah, J., Bikshandi, G., Zhang, Y., Thakkar, V., Ramani, P., & Dao, T. (2024). FlashAttention-3: Fast and accurate attention with asynchrony and low-precision. *NeurIPS 2024*. arXiv:2407.08608.
- Zadouri, T., Hoehnerbach, M., Shah, J., Liu, T., Thakkar, V., & Dao, T. (2026). FlashAttention-4. arXiv:2603.05451. *[preprint]*
- Kwon, W., et al. (2023). Efficient memory management for large language model serving with PagedAttention. *SOSP 2023*. arXiv:2309.06180.
- Child, R., Gray, S., Radford, A., & Sutskever, I. (2019). Generating long sequences with sparse Transformers. arXiv:1904.10509. *[preprint]*
- Beltagy, I., Peters, M. E., & Cohan, A. (2020). Longformer: The long-document Transformer. arXiv:2004.05150.
- Zaheer, M., et al. (2020). Big Bird: Transformers for longer sequences. *NeurIPS 2020*. arXiv:2007.14062.
- Jiang, A. Q., et al. (2023). Mistral 7B. arXiv:2310.06825. *[preprint]*
- Gemma Team. (2025). Gemma 3 technical report. arXiv:2503.19786. *[preprint]*
- Yuan, J., Gao, H., Dai, D., et al. (2025). Native sparse attention: Hardware-aligned and natively trainable sparse attention. *ACL 2025 (Long)*, 23078–23097.
- Katharopoulos, A., Vyas, A., Pappas, N., & Fleuret, F. (2020). Transformers are RNNs: Fast autoregressive Transformers with linear attention. *ICML 2020*. arXiv:2006.16236.
- Gu, A., & Dao, T. (2024). Mamba: Linear-time sequence modeling with selective state spaces. *COLM 2024*. arXiv:2312.00752.
- Dao, T., & Gu, A. (2024). Transformers are SSMs: Generalized models and efficient algorithms through structured state space duality. *ICML 2024*. arXiv:2405.21060.
- Lieber, O., et al. (2024). Jamba: A hybrid Transformer–Mamba language model. arXiv:2403.19887. *[preprint]*
- Qwen Team. (2025). Qwen3 technical report. arXiv:2505.09388. *[preprint]*; Qwen3-Next-80B-A3B model card. *[preprint]*
- Meta. (2024). The Llama 3 herd of models. arXiv:2407.21783. *[preprint]*
- Shazeer, N. (2020). GLU variants improve Transformer. arXiv:2002.05202. *[preprint]*
- Dehghani, M., et al. (2023). Scaling vision Transformers to 22 billion parameters. *ICML 2023*. arXiv:2302.05442.
- Qiu, Z., Wang, Z., Zheng, B., et al. (2025). Gated attention for large language models: Non-linearity, sparsity, and attention-sink-free. *NeurIPS 2025*. arXiv:2505.06708.
- Xiao, G., Tian, Y., Chen, B., Han, S., & Lewis, M. (2024). Efficient streaming language models with attention sinks. *ICLR 2024*. arXiv:2309.17453.
- Liu, N. F., Lin, K., Hewitt, J., Paranjape, A., Bevilacqua, M., Petroni, F., & Liang, P. (2024). Lost in the middle: How language models use long contexts. *TACL*, 12. arXiv:2307.03172.
- Ye, T., Dong, L., Xia, Y., Sun, Y., Zhu, Y., Huang, G., & Wei, F. (2025). Differential Transformer. *ICLR 2025*. arXiv:2410.05258.
- Hu, E. J., Shen, Y., Wallis, P., et al. (2022). LoRA: Low-rank adaptation of large language models. *ICLR 2022*.
- Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2019). BERT: Pre-training of deep bidirectional Transformers for language understanding. *NAACL-HLT 2019*, 4171–4186.
- Raffel, C., et al. (2020). Exploring the limits of transfer learning with a unified text-to-text Transformer. *JMLR*, 21(140), 1–67.

### Tiếng Việt
- Nguyen, D. Q., & Nguyen, A. T. (2020). PhoBERT: Pre-trained language models for Vietnamese. *Findings of EMNLP 2020*, 1037–1042.
- Vu, T., Nguyen, D. Q., Nguyen, D. Q., Dras, M., & Johnson, M. (2018). VnCoreNLP: A Vietnamese natural language processing toolkit. *NAACL 2018 (Demonstrations)*, 56–60.
- Nguyen, D. Q., Nguyen, D. Q., Vu, T., Dras, M., & Johnson, M. (2018). A fast and accurate Vietnamese word segmenter. *LREC 2018*.
- Nguyen, N. H., Nguyen, D. T., & Nguyen, N. L.-T. (2025). Vietnamese words are not constructed from syllables: Rethinking the role of word segmentation in NLP for Vietnamese texts. *AAAI 2025*.
- Phan, L., Tran, H., Nguyen, H., & Trinh, T. H. (2022). ViT5: Pretrained text-to-text Transformer for Vietnamese language generation. *NAACL 2022 Student Research Workshop*.
- Tran, C. D., et al. (2023). ViDeBERTa: A powerful pre-trained language model for Vietnamese. *Findings of EACL 2023*, 1071–1078.
- Nguyen, D. Q., Nguyen, L. T., Tran, C., Nguyen, D. N., Phung, D., & Bui, H. (2023). PhoGPT: Generative pre-training for Vietnamese. arXiv:2311.02945. *[preprint]*
- Viet-Mistral. Vistral-7B-Chat (model card). https://huggingface.co/Viet-Mistral/Vistral-7B-Chat *[preprint]*
- Nguyen, Q.-N., Phan, T. C., Nguyen, D.-V., & Nguyen, K. V. (2023). ViSoBERT: A pre-trained language model for Vietnamese social media text processing. *EMNLP 2023*.
- Petrov, A., La Malfa, E., Torr, P. H. S., & Bibi, A. (2023). Language model tokenizers introduce unfairness between languages. *NeurIPS 2023*. arXiv:2305.15425.
- Luong, M.-T., & Manning, C. D. (2015). Stanford neural machine translation systems for spoken language domains. *IWSLT 2015*.
- Doan, L., Nguyen, L. T., Tran, N. L., Hoang, T., & Nguyen, D. Q. (2021). PhoMT: A high-quality and large-scale benchmark dataset for Vietnamese–English machine translation. *EMNLP 2021*.
- Dang, T. D. A., & Nguyen, T. T. T. (2020). TDP – A hybrid diacritic restoration with Transformer decoder. *PACLIC 2020*. https://aclanthology.org/2020.paclic-1.9/
- Wang, Y., Cui, L., & Zhang, Y. (2020). Does Chinese BERT encode word structure? *COLING 2020*.

---
*© Tài liệu giảng dạy học phần "Deep Learning trong NLP" — TS. Hoàng Văn Quý. Phần cài đặt kế thừa The Annotated Transformer (Harvard NLP) và phần trực giác kế thừa 3Blue1Brown; vui lòng giữ nguyên ghi nguồn khi chia sẻ.*